In [1]:
# If required, uncomment:
# %pip install pandas numpy matplotlib seaborn scipy openpyxl

import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

sns.set_theme(style="whitegrid")

In [2]:
from pathlib import Path

FILE_CHANGE_EXPENSES = Path(
    r"E:\Bond\Data\osc_sheet_Change in Stock__Total Expenses on combined operations.csv"
)

FILE_PL_COMPOSITION = Path(
    r"E:\Bond\Data\osc_sheet_P&L Statement Composition.csv"
)

# Choose reporting basis
REPORTING_BASIS = "Consolidated"

# Output directory
OUTPUT_DIR = Path("financial_analysis_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Check files
for file_path in [FILE_CHANGE_EXPENSES, FILE_PL_COMPOSITION]:
    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

print("Both input files found successfully.")
print("Change/Expenses file:", FILE_CHANGE_EXPENSES)
print("P&L file:", FILE_PL_COMPOSITION)
print("Output folder:", OUTPUT_DIR.resolve())

Both input files found successfully.
Change/Expenses file: E:\Bond\Data\osc_sheet_Change in Stock__Total Expenses on combined operations.csv
P&L file: E:\Bond\Data\osc_sheet_P&L Statement Composition.csv
Output folder: E:\Bond\Data\financial_analysis_outputs


In [3]:
raw_change = pd.read_csv(FILE_CHANGE_EXPENSES, header=None, dtype=str, low_memory=False)
raw_pl = pd.read_csv(FILE_PL_COMPOSITION, header=None, dtype=str, low_memory=False)

print("Change in Stock & Expenses raw shape:", raw_change.shape)
print("P&L Composition raw shape:", raw_pl.shape)

display(raw_change.iloc[:7, :8])
display(raw_pl.iloc[:7, :8])

Change in Stock & Expenses raw shape: (13957, 280)
P&L Composition raw shape: (13168, 81)


,0,1,2,3,4,5,6,7
0,NaN,CMIE Expr,CMIE Expr,CMIE Expr,CMIE Expr,CMIE Expr,CMIE Expr,CMIE Expr
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Standardised Annual Consolidated (Ind AS),Standardised Annual Consolidated (Ind AS),Standardised Annual Consolidated (Ind AS),Standardised Annual Consolidated (Ind AS),Standardised Annual Consolidated (Ind AS),Standardised Annual Consolidated (Ind AS),Standardised Annual Consolidated (Ind AS)
3,NaN,Rs. Crore,Rs. Crore,Rs. Crore,Rs. Crore,Rs. Crore,Rs. Crore,Rs. Crore
4,NaN,Mar 2026,Mar 2026,Mar 2026,Mar 2026,Mar 2026,Mar 2026,Mar 2026
5,Company Name,Change in stock,Total Income from continued operations,Change in stock of finished goods (including i...,Income from non-financial services,Change in stock of wip and semifinished goods ...,Manufacturing revenue,Other adjustments in change in inventories
6,2 X 2 Logistics Pvt. Ltd.,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,0,1,2,3,4,5,6,7
0,NaN,CMIE Expr,CMIE Expr,CMIE Expr,CMIE Expr,CMIE Expr,CMIE Expr,CMIE Expr
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Standardised Annual Consolidated (Ind AS),Standardised Annual Consolidated (Ind AS),Standardised Annual Consolidated (Ind AS),Standardised Annual Consolidated (Ind AS),Standardised Annual Consolidated (Ind AS),Standardised Annual Consolidated (Ind AS),Standardised Annual Consolidated (Ind AS)
3,NaN,Rs. Crore,Rs. Crore,Rs. Crore,Rs. Crore,Rs. Crore,Rs. Crore,Rs. Crore
4,NaN,Mar 2026,Mar 2026,Mar 2026,Mar 2026,Mar 2026,Mar 2026,Mar 2026
5,Company Name,Total Income from continued operations,Income from non-financial services,Manufacturing revenue,Trading revenue and non-financial services,Sales of goods,Job-work income,Income from repairs & maintenance including af...
6,2 X 2 Logistics Pvt. Ltd.,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# ============================================================
# PART 1: PARSE AND SEPARATE FINANCIAL DATASETS
# ============================================================

def parse_financial_dataset(raw_df, dataset_name):

    print("\nProcessing:", dataset_name)

    # Extract metadata
    statement_types = (
        raw_df.iloc[2, 1:]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    financial_variables = (
        raw_df.iloc[5, 1:]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    # Actual company observations start from row 7
    company_data = raw_df.iloc[6:, :].copy()

    company_data = company_data.rename(
        columns={0: "Company Name"}
    )

    company_data["Company Name"] = (
        company_data["Company Name"]
        .astype("string")
        .str.strip()
    )

    results = {}

    for reporting_type in ["Consolidated", "Standalone"]:

        # Identify columns belonging to the reporting type
        positions = [
            i + 1
            for i, value in enumerate(statement_types)
            if reporting_type.lower() in value.lower()
        ]

        if len(positions) == 0:

            print(f"{reporting_type} data not found.")
            continue

        # Extract financial variable names
        columns = [
            financial_variables.iloc[pos - 1]
            for pos in positions
        ]

        # Handle duplicate financial variable names
        unique_columns = []
        counts = {}

        for col in columns:

            if not col:
                col = "Unknown Financial Variable"

            counts[col] = counts.get(col, 0) + 1

            if counts[col] > 1:
                unique_columns.append(
                    f"{col}_duplicate_{counts[col]}"
                )
            else:
                unique_columns.append(col)

        # Select relevant observations
        temp = company_data[
            ["Company Name"] + positions
        ].copy()

        temp.columns = [
            "Company Name"
        ] + unique_columns

        # Convert financial variables to numeric
        for col in unique_columns:

            temp[col] = (
                temp[col]
                .astype("string")
                .str.replace(",", "", regex=False)
                .str.strip()
            )

            temp[col] = pd.to_numeric(
                temp[col],
                errors="coerce"
            )

        # Remove invalid company names
        temp = temp[
            temp["Company Name"].notna()
            & (temp["Company Name"] != "")
        ]

        # Remove duplicate companies
        temp = temp.drop_duplicates(
            subset=["Company Name"],
            keep="first"
        )

        temp = temp.reset_index(drop=True)

        results[reporting_type] = temp

        print("\nReporting Type:", reporting_type)
        print("Companies:", temp.shape[0])
        print("Financial Variables:", len(unique_columns))
        print("Dataset Shape:", temp.shape)

    return results


# Parse both datasets

change_data = parse_financial_dataset(
    raw_change,
    "Change in Stock and Total Expenses"
)

pl_data = parse_financial_dataset(
    raw_pl,
    "P&L Statement Composition"
)


# Separate reporting structures

change_consolidated = change_data["Consolidated"]
change_standalone = change_data["Standalone"]

pl_consolidated = pl_data["Consolidated"]
pl_standalone = pl_data["Standalone"]


print("\nPARSING COMPLETED")


Processing: Change in Stock and Total Expenses

Reporting Type: Consolidated
Companies: 13951
Financial Variables: 141
Dataset Shape: (13951, 142)

Reporting Type: Standalone
Companies: 13951
Financial Variables: 138
Dataset Shape: (13951, 139)

Processing: P&L Statement Composition

Reporting Type: Consolidated
Companies: 13162
Financial Variables: 41
Dataset Shape: (13162, 42)

Reporting Type: Standalone
Companies: 13162
Financial Variables: 39
Dataset Shape: (13162, 40)

PARSING COMPLETED


In [5]:
# ============================================================
# PART 2: COMPANY ALIGNMENT AND FINANCIAL VARIABLE PROFILING
# ============================================================

# ------------------------------------------------------------
# 1. Select Consolidated datasets
# ------------------------------------------------------------

change_df = change_consolidated.copy()
pl_df = pl_consolidated.copy()

print("Change/Expenses shape:", change_df.shape)
print("P&L Composition shape:", pl_df.shape)


# ------------------------------------------------------------
# 2. Normalize company names for matching
# ------------------------------------------------------------

def normalize_company_name(name):

    name = str(name).upper().strip()

    # Normalize whitespace
    name = re.sub(r"\s+", " ", name)

    # Normalize punctuation
    name = re.sub(r"[.,]", "", name)

    return name


change_df["Company_Key"] = (
    change_df["Company Name"]
    .apply(normalize_company_name)
)

pl_df["Company_Key"] = (
    pl_df["Company Name"]
    .apply(normalize_company_name)
)


# ------------------------------------------------------------
# 3. Identify company coverage
# ------------------------------------------------------------

change_companies = set(change_df["Company_Key"])
pl_companies = set(pl_df["Company_Key"])

common_companies = change_companies.intersection(
    pl_companies
)

only_change = change_companies - pl_companies
only_pl = pl_companies - change_companies

print("\nCOMPANY COVERAGE ANALYSIS")
print("-" * 60)

print("Companies in Change/Expenses:", len(change_companies))
print("Companies in P&L:", len(pl_companies))
print("Common companies:", len(common_companies))
print("Only in Change/Expenses:", len(only_change))
print("Only in P&L:", len(only_pl))

print(
    "Overlap percentage:",
    round(
        len(common_companies)
        / len(change_companies.union(pl_companies))
        * 100,
        2
    ),
    "%"
)


# ------------------------------------------------------------
# 4. Merge datasets using exact normalized company keys
# ------------------------------------------------------------

change_features = change_df.drop(
    columns=["Company Name"]
).rename(
    columns={
        col: f"CHANGE__{col}"
        for col in change_df.columns
        if col not in ["Company_Key"]
    }
)

pl_features = pl_df.drop(
    columns=["Company Name"]
).rename(
    columns={
        col: f"PL__{col}"
        for col in pl_df.columns
        if col not in ["Company_Key"]
    }
)

financial_df = change_df[
    ["Company_Key", "Company Name"]
].merge(
    pl_features,
    on="Company_Key",
    how="inner",
    validate="one_to_one"
)

financial_df = financial_df.merge(
    change_features,
    on="Company_Key",
    how="inner",
    validate="one_to_one"
)

print("\nUNIFIED FINANCIAL DATASET")
print("-" * 60)

print("Shape:", financial_df.shape)
print("Number of companies:", financial_df["Company_Key"].nunique())
print("Number of financial features:",
      financial_df.shape[1] - 2)

display(financial_df.head())


# ------------------------------------------------------------
# 5. Create financial variable catalogue
# ------------------------------------------------------------

catalogue = []

for dataset_name, df in [
    ("Change_Expenses", change_df),
    ("PL_Composition", pl_df)
]:

    for col in df.columns:

        if col in ["Company Name", "Company_Key"]:
            continue

        values = df[col].dropna()

        catalogue.append({

            "Dataset": dataset_name,

            "Variable": col,

            "Data_Type": str(df[col].dtype),

            "Non_Missing": len(values),

            "Missing": df[col].isna().sum(),

            "Missing_Percentage":
                round(df[col].isna().mean() * 100, 2),

            "Unique_Values": df[col].nunique(),

            "Mean": values.mean() if len(values) else np.nan,

            "Median": values.median() if len(values) else np.nan,

            "Std": values.std() if len(values) > 1 else np.nan,

            "Minimum": values.min() if len(values) else np.nan,

            "Maximum": values.max() if len(values) else np.nan,

            "Zero_Count": (values == 0).sum(),

            "Negative_Count": (values < 0).sum()

        })


variable_catalogue = pd.DataFrame(catalogue)

variable_catalogue = variable_catalogue.sort_values(
    by="Missing_Percentage",
    ascending=True
).reset_index(drop=True)


# ------------------------------------------------------------
# 6. Display variable catalogue
# ------------------------------------------------------------

print("\nFINANCIAL VARIABLE CATALOGUE")
print("-" * 60)

print("Total financial variables:", len(variable_catalogue))

display(variable_catalogue)


# ------------------------------------------------------------
# 7. Identify high-quality and low-quality variables
# ------------------------------------------------------------

print("\nVARIABLE QUALITY SUMMARY")
print("-" * 60)

print(
    "Variables with less than 20% missing:",
    (variable_catalogue["Missing_Percentage"] < 20).sum()
)

print(
    "Variables with less than 50% missing:",
    (variable_catalogue["Missing_Percentage"] < 50).sum()
)

print(
    "Variables with more than 80% missing:",
    (variable_catalogue["Missing_Percentage"] > 80).sum()
)

print(
    "Completely empty variables:",
    (variable_catalogue["Non_Missing"] == 0).sum()
)


# ------------------------------------------------------------
# 8. Save intermediate outputs
# ------------------------------------------------------------

financial_df.to_csv(
    OUTPUT_DIR / "unified_consolidated_financial_data.csv",
    index=False
)

variable_catalogue.to_csv(
    OUTPUT_DIR / "financial_variable_catalogue.csv",
    index=False
)

print("\nSaved:")
print("1. unified_consolidated_financial_data.csv")
print("2. financial_variable_catalogue.csv")

Change/Expenses shape: (13951, 142)
P&L Composition shape: (13162, 42)

COMPANY COVERAGE ANALYSIS
------------------------------------------------------------
Companies in Change/Expenses: 13951
Companies in P&L: 13162
Common companies: 13162
Only in Change/Expenses: 789
Only in P&L: 0
Overlap percentage: 94.34 %

UNIFIED FINANCIAL DATASET
------------------------------------------------------------
Shape: (13162, 184)
Number of companies: 13162
Number of financial features: 182


,Company_Key,Company Name,PL__Total Income from continued operations,PL__Income from non-financial services,PL__Manufacturing revenue,PL__Trading revenue and non-financial services,PL__Sales of goods,PL__Job-work income,PL__Income from repairs & maintenance including after-sales service income,PL__Construction income,"PL__Sale of electricity, gas and water",PL__Fiscal benefits,"PL__Of which: Export incentives including duty draw back, etc.",PL__Of which: Trading income,PL__Of which: Royalty income,PL__Of which: Rent/Operating lease rent income,PL__Income from financial services,PL__Fee based financial services income,PL__Of which: Brokerage & commission fees,PL__Interest income,PL__Interest income from banks,PL__Interest income of cos other than banks from investments,PL__Interest income of cos other than banks from loans and advances,PL__Dividend income,PL__Of which: Dividend from group companies,PL__Share of profit in partnership and JV firms,PL__Profit on securitisation/assignment of assets and loans,PL__Income from treasury operations,PL__Gain on financial instruments,PL__Gain relating to forex transactions,PL__Other financial services income,PL__Other income,PL__Of which: Income from carbon credits,PL__Material/exceptional income,PL__Bad debts recovered,PL__Provisions/impairment and credit balances written back,PL__Gain on corporate and debt restructuring (inclg. one time debt waiver),"PL__Profit on sale of investment in subsidiary, associates & JV","PL__Fair value gain on retained interest in a subsidiary, associate or JV",PL__Fair value gain on step acquisition of a subsidiary,PL__Gain on disposal of non-current non-financial assets,PL__Insurance claims,PL__Less: Income capitalised and transferred to DRE,CHANGE__Change in stock,CHANGE__Total Income from continued operations,CHANGE__Change in stock of finished goods (including in transit),CHANGE__Income from non-financial services,CHANGE__Change in stock of wip and semifinished goods (including in transit),CHANGE__Manufacturing revenue,CHANGE__Other adjustments in change in inventories,...,CHANGE__Consultancy fees,CHANGE__Consultancy fees to auditors,CHANGE__Consultancy fees to others,CHANGE__Other professional services,CHANGE__Of which: Cost audit fees,CHANGE__Of which: Legal charges,CHANGE__Non-executive directors' fees & commission,CHANGE__Selling & distribution expenses,CHANGE__Marketing expenses,CHANGE__Rebates & discount expenses,CHANGE__Sales promotion expenses,CHANGE__Advertising expenses,CHANGE__Distribution expenses (including outward freight),CHANGE__Travel expenses,CHANGE__Communications expenses,CHANGE__Telephone expenses,CHANGE__Postage & courier,"CHANGE__Expenses on data centres, web hosting and co hosting","CHANGE__Expenses on vsats, satellite links",CHANGE__Expenses on isps for internet services,CHANGE__Software charges,CHANGE__IT enabled services charges,CHANGE__Miscellaneous expenditure,CHANGE__Donations,CHANGE__Social and community expenses (including CSR exp),CHANGE__Environment and pollution control related expenses,CHANGE__Penalties on direct taxes,CHANGE__Research & development expenses,CHANGE__Penalties on taxes,CHANGE__Other operational expenses of non-financial services enterprises,CHANGE__Financial services expenses,CHANGE__Fee based financial services expenses,CHANGE__Bank charges and commission,CHANGE__Guarantee fees and commission,CHANGE__Fund based financial services expenses,CHANGE__Interest expense,CHANGE__Interest on long term borrowings / convertible borrowings,CHANGE__Interest on short term borrowings / bank overdrafts / revolving credit facility,CHANGE__Other interest expense (term not specified),CHANGE__Interest on finance lease,CHANGE__Less: Interest capitalised,CHANGE__Premium/discount on debt instruments,CHANGE__Other borrowing costs,CHANGE__Bill discounting charges,CHANGE__Other fund based financial services expenses,CHANGE__Of which: Share of loss in partnership and JV firms,CHANGE__Of which: Loss on securitisation/assignment o


FINANCIAL VARIABLE CATALOGUE
------------------------------------------------------------
Total financial variables: 182


,Dataset,Variable,Data_Type,Non_Missing,Missing,Missing_Percentage,Unique_Values,Mean,Median,Std,Minimum,Maximum,Zero_Count,Negative_Count
0,PL_Composition,Total Income from continued operations,Float64,1772,11390,86.54,1769,"9,144.18","1,044.07","52,110.46",0.01,"1,204,916.00",0,0
1,PL_Composition,Income from financial services,Float64,1748,11414,86.72,1425,217.39,14.02,"1,615.64",0.01,"46,387.85",0,0
2,PL_Composition,Income from non-financial services,Float64,1740,11422,86.78,1734,"8,940.12","1,007.75","51,376.57",0.01,"1,175,919.00",0,0
3,PL_Composition,Interest income,Float64,1730,11432,86.86,1178,117.66,5.58,935.30,0.01,"25,043.19",0,0
4,Change_Expenses,Ordinary and non-exceptional expenses,Float64,1776,12175,87.27,1767,"8,276.40",934.57,"48,223.87",0.22,"1,091,905.00",0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
177,Change_Expenses,Fair value gain on step acquisition of a subsi...,Float64,2,13949,99.99,2,223.23,223.23,272.19,30.76,415.70,0,0
178,Change_Expenses,Of which: Service tax,Float64,1,13950,99.99,1,1.28,1.28,NaN,1.28,1.28,0,0
179,Change_Expenses,Of which: Loss on securitisation/assignment of...,Float64,0,13951,100.00,0,NaN,NaN,NaN,NaN,NaN,0,0
180,Change_Expenses,Penalties on direct taxes,Float64,0,13951,100.00,0,NaN,NaN,NaN,NaN,NaN,0,0



VARIABLE QUALITY SUMMARY
------------------------------------------------------------
Variables with less than 20% missing: 0
Variables with less than 50% missing: 0
Variables with more than 80% missing: 182
Completely empty variables: 3

Saved:
1. unified_consolidated_financial_data.csv
2. financial_variable_catalogue.csv


In [6]:
# ============================================================
# PART 3: FINANCIAL VARIABLE EXTRACTION AND NLP PREPROCESSING
# ============================================================

import re
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Extract financial variables from both datasets
# ------------------------------------------------------------

pl_variables = [
    col for col in pl_df.columns
    if col not in ["Company Name", "Company_Key"]
]

change_variables = [
    col for col in change_df.columns
    if col not in ["Company Name", "Company_Key"]
]

variable_records = []

for variable in pl_variables:

    variable_records.append({
        "Original_Variable": variable,
        "Dataset": "PL_Composition"
    })

for variable in change_variables:

    variable_records.append({
        "Original_Variable": variable,
        "Dataset": "Change_Expenses"
    })

nlp_variables = pd.DataFrame(variable_records)

print("Total variable records:", len(nlp_variables))
print("Unique financial labels:",
      nlp_variables["Original_Variable"].nunique())

# ------------------------------------------------------------
# 2. Normalize financial variable names
# ------------------------------------------------------------

def normalize_financial_text(text):

    text = str(text).lower().strip()

    # Remove duplicate suffixes introduced during parsing
    text = re.sub(r"_duplicate_\d+$", "", text)

    # Remove punctuation
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()


nlp_variables["Normalized_Variable"] = (
    nlp_variables["Original_Variable"]
    .apply(normalize_financial_text)
)

# ------------------------------------------------------------
# 3. Detect accounting qualifiers
# ------------------------------------------------------------

def identify_qualifier(text):

    text = text.lower()

    if text.startswith("of which"):
        return "Subcomponent"

    if text.startswith("less"):
        return "Deduction"

    if "exceptional" in text:
        return "Exceptional"

    if "adjustment" in text:
        return "Adjustment"

    return "Primary"


nlp_variables["Accounting_Qualifier"] = (
    nlp_variables["Original_Variable"]
    .apply(identify_qualifier)
)

# ------------------------------------------------------------
# 4. Tokenize financial labels
# ------------------------------------------------------------

def tokenize_financial_text(text):

    tokens = text.split()

    return tokens


nlp_variables["Tokens"] = (
    nlp_variables["Normalized_Variable"]
    .apply(tokenize_financial_text)
)

nlp_variables["Token_Count"] = (
    nlp_variables["Tokens"].apply(len)
)

# ------------------------------------------------------------
# 5. Inspect extracted NLP dataset
# ------------------------------------------------------------

print("\nFINANCIAL NLP DATASET")
display(nlp_variables.head(30))

print("\nAccounting qualifier distribution:")
display(
    nlp_variables["Accounting_Qualifier"]
    .value_counts()
)

print("\nDataset-wise variable distribution:")
display(
    nlp_variables.groupby("Dataset")
    .agg(
        Total_Variables=("Original_Variable", "count"),
        Unique_Variables=("Normalized_Variable", "nunique")
    )
)

# ------------------------------------------------------------
# 6. Save
# ------------------------------------------------------------

nlp_variables.to_csv(
    OUTPUT_DIR / "financial_nlp_variable_dictionary.csv",
    index=False
)

print("\nSaved financial_nlp_variable_dictionary.csv")

Total variable records: 182
Unique financial labels: 141

FINANCIAL NLP DATASET


,Original_Variable,Dataset,Normalized_Variable,Accounting_Qualifier,Tokens,Token_Count
0,Total Income from continued operations,PL_Composition,total income from continued operations,Primary,"[total, income, from, continued, operations]",5
1,Income from non-financial services,PL_Composition,income from non financial services,Primary,"[income, from, non, financial, services]",5
2,Manufacturing revenue,PL_Composition,manufacturing revenue,Primary,"[manufacturing, revenue]",2
3,Trading revenue and non-financial services,PL_Composition,trading revenue and non financial services,Primary,"[trading, revenue, and, non, financial, services]",6
4,Sales of goods,PL_Composition,sales of goods,Primary,"[sales, of, goods]",3
5,Job-work income,PL_Composition,job work income,Primary,"[job, work, income]",3
6,Income from repairs & maintenance including af...,PL_Composition,income from repairs maintenance including afte...,Primary,"[income, from, repairs, maintenance, including...",9
7,Construction income,PL_Composition,construction income,Primary,"[construction, income]",2
8,"Sale of electricity, gas and water",PL_Composition,sale of electricity gas and water,Primary,"[sale, of, electricity, gas, and, water]",6
9,Fiscal benefits,PL_Composition,fiscal benefits,Primary,"[fiscal, benefits]",2



Accounting qualifier distribution:


Accounting_Qualifier
Primary         154
Subcomponent     19
Deduction         5
Exceptional       3
Adjustment        1
Name: count, dtype: int64


Dataset-wise variable distribution:


,Total_Variables,Unique_Variables
Dataset,,
Change_Expenses,141,141
PL_Composition,41,41



Saved financial_nlp_variable_dictionary.csv


In [7]:
# ============================================================
# PART 4: HIERARCHICAL FINANCIAL ONTOLOGY
# ============================================================

import re
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Define financial ontology
# ------------------------------------------------------------

FINANCIAL_ONTOLOGY = {

    "Revenue": {
        "Operating Revenue": [
            "manufacturing revenue",
            "trading revenue",
            "sales of goods",
            "job work income",
            "construction income",
            "repairs maintenance",
            "sale of electricity",
            "non financial services"
        ],

        "Financial Revenue": [
            "interest income",
            "dividend income",
            "financial services income",
            "brokerage",
            "commission fees",
            "treasury operations",
            "financial instruments",
            "forex transactions",
            "royalty income"
        ],

        "Other Revenue": [
            "other income",
            "fiscal benefits",
            "insurance claims",
            "bad debts recovered",
            "export incentives",
            "carbon credits",
            "rent income",
            "lease rent income"
        ]
    },

    "Expenses": {
        "Material and Production Costs": [
            "raw material",
            "purchase of stock",
            "power fuel",
            "manufacturing expenses",
            "production expenses",
            "stores and spares",
            "packing materials"
        ],

        "Employee Costs": [
            "salaries",
            "wages",
            "employee benefits",
            "staff welfare",
            "gratuity",
            "provident fund",
            "pension"
        ],

        "Operating and Administrative Costs": [
            "administrative expenses",
            "selling expenses",
            "distribution expenses",
            "marketing expenses",
            "advertising expenses",
            "travel expenses",
            "communication expenses",
            "telephone expenses",
            "postage",
            "courier",
            "software charges",
            "consultancy fees",
            "professional services",
            "legal charges",
            "audit fees",
            "donations",
            "csr",
            "research development",
            "miscellaneous expenditure",
            "other operational expenses"
        ],

        "Finance Costs": [
            "interest expense",
            "borrowing costs",
            "interest on borrowings",
            "bank charges",
            "bill discounting",
            "finance lease",
            "debt instruments",
            "guarantee fees"
        ],

        "Non-Cash Expenses": [
            "depreciation",
            "amortisation",
            "impairment",
            "provisions"
        ],

        "Tax and Regulatory Costs": [
            "tax expenses",
            "service tax",
            "penalties",
            "duties",
            "statutory levies"
        ]
    },

    "Inventory": {
        "Inventory Changes": [
            "change in stock",
            "change in inventories",
            "finished goods",
            "work in progress",
            "wip",
            "semifinished goods",
            "inventory adjustment"
        ]
    },

    "Gains and Losses": {
        "Asset and Investment Gains": [
            "gain on disposal",
            "profit on sale of investment",
            "fair value gain",
            "gain on step acquisition",
            "gain on restructuring",
            "profit on securitisation"
        ],

        "Financial Gains and Losses": [
            "gain relating to forex",
            "loss relating to forex",
            "gain on financial instruments",
            "loss on financial instruments",
            "share of profit",
            "share of loss"
        ]
    },

    "Accounting Adjustments": {
        "Capitalisation and Transfers": [
            "income capitalised",
            "interest capitalised",
            "transferred to dre"
        ],

        "Exceptional and Other Adjustments": [
            "exceptional items",
            "one time debt waiver",
            "other adjustments"
        ]
    }
}


# ------------------------------------------------------------
# 2. Classification function
# ------------------------------------------------------------

def classify_financial_variable(variable):

    text = normalize_financial_text(variable)

    # Remove accounting qualifier for concept classification
    text = re.sub(
        r"^(of which|less)\s+",
        "",
        text
    )

    # Search specific ontology categories
    for domain, categories in FINANCIAL_ONTOLOGY.items():

        for category, keywords in categories.items():

            for keyword in keywords:

                if keyword in text:

                    return {
                        "Financial_Domain": domain,
                        "Financial_Category": category,
                        "Classification_Matched": keyword,
                        "Classification_Status": "Rule matched"
                    }

    return {
        "Financial_Domain": "Unclassified",
        "Financial_Category": "Unclassified",
        "Classification_Matched": None,
        "Classification_Status": "Requires review"
    }


# ------------------------------------------------------------
# 3. Apply ontology classification
# ------------------------------------------------------------

classification_results = nlp_variables[
    "Original_Variable"
].apply(classify_financial_variable)

classification_df = pd.DataFrame(
    classification_results.tolist()
)

ontology_df = pd.concat(
    [
        nlp_variables.reset_index(drop=True),
        classification_df.reset_index(drop=True)
    ],
    axis=1
)


# ------------------------------------------------------------
# 4. Create hierarchical accounting identifiers
# ------------------------------------------------------------

ontology_df["Ontology_ID"] = (
    ontology_df["Financial_Domain"].str.upper()
    + "_"
    + ontology_df["Financial_Category"].str.upper()
    + "_"
    + ontology_df.index.astype(str)
)

ontology_df["Financial_Hierarchy"] = (
    ontology_df["Financial_Domain"]
    + " > "
    + ontology_df["Financial_Category"]
    + " > "
    + ontology_df["Normalized_Variable"]
)


# ------------------------------------------------------------
# 5. Generate ontology statistics
# ------------------------------------------------------------

print("\nFINANCIAL DOMAIN DISTRIBUTION")

display(
    ontology_df["Financial_Domain"]
    .value_counts()
    .to_frame("Variable_Count")
)

print("\nFINANCIAL CATEGORY DISTRIBUTION")

display(
    ontology_df.groupby(
        ["Financial_Domain", "Financial_Category"]
    ).size()
    .reset_index(name="Variable_Count")
    .sort_values("Variable_Count", ascending=False)
)

print("\nCLASSIFICATION STATUS")

display(
    ontology_df["Classification_Status"]
    .value_counts()
)


# ------------------------------------------------------------
# 6. Identify unclassified variables
# ------------------------------------------------------------

unclassified = ontology_df[
    ontology_df["Financial_Domain"] == "Unclassified"
]

print("\nUNCLASSIFIED FINANCIAL VARIABLES:", len(unclassified))

display(
    unclassified[
        ["Original_Variable", "Dataset", "Accounting_Qualifier"]
    ]
)


# ------------------------------------------------------------
# 7. Save ontology
# ------------------------------------------------------------

ontology_df.to_csv(
    OUTPUT_DIR / "financial_ontology.csv",
    index=False
)

print("\nSaved financial_ontology.csv")


FINANCIAL DOMAIN DISTRIBUTION


,Variable_Count
Financial_Domain,
Revenue,62
Unclassified,53
Expenses,44
Gains and Losses,13
Accounting Adjustments,5
Inventory,5



FINANCIAL CATEGORY DISTRIBUTION


,Financial_Domain,Financial_Category,Variable_Count
14,Unclassified,Unclassified,53
11,Revenue,Financial Revenue,27
12,Revenue,Operating Revenue,21
6,Expenses,Operating and Administrative Costs,20
13,Revenue,Other Revenue,14
8,Gains and Losses,Asset and Investment Gains,10
3,Expenses,Finance Costs,8
2,Expenses,Employee Costs,5
10,Inventory,Inventory Changes,5
4,Expenses,Material and Production Costs,4



CLASSIFICATION STATUS


Classification_Status
Rule matched       129
Requires review     53
Name: count, dtype: int64


UNCLASSIFIED FINANCIAL VARIABLES: 53


,Original_Variable,Dataset,Accounting_Qualifier
0,Total Income from continued operations,PL_Composition,Primary
11,Of which: Trading income,PL_Composition,Subcomponent
14,Income from financial services,PL_Composition,Primary
22,Of which: Dividend from group companies,PL_Composition,Subcomponent
31,Material/exceptional income,PL_Composition,Exceptional
42,Total Income from continued operations,Change_Expenses,Primary
50,Total expenses of continued operations,Change_Expenses,Primary
52,Total expenses excluding direct taxes,Change_Expenses,Primary
56,Ordinary and non-exceptional expenses,Change_Expenses,Exceptional
62,"Stores, spares, tools consumed",Change_Expenses,Primary



Saved financial_ontology.csv


In [8]:
# ============================================================
# PART 5: REFINED FINANCIAL ONTOLOGY
# ============================================================

import re
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Define priority-based financial classification rules
# ------------------------------------------------------------

ONTOLOGY_RULES = [

    # TOTAL FINANCIAL AGGREGATES
    (
        r"\btotal income\b",
        "Revenue",
        "Total Revenue"
    ),
    (
        r"\btotal expenses?\b|\bordinary and non exceptional expenses\b",
        "Expenses",
        "Total Expenses"
    ),

    # INVENTORY
    (
        r"change in stock|change in inventories|finished goods|"
        r"work in progress|\bwip\b|semifinished goods|inventory",
        "Inventory",
        "Inventory Changes"
    ),

    # MATERIAL AND PRODUCTION COSTS
    (
        r"raw material|stores spares|tools consumed|"
        r"packaging|packing expenses|purchase of stock|"
        r"power and fuel|power fuel|outsourced industrial jobs|"
        r"manufacturing expenses|production expenses",
        "Expenses",
        "Material and Production Costs"
    ),

    # EMPLOYEE COSTS
    (
        r"salaries|wages|bonus|ex gratia|gratuities|"
        r"superannuation|employee benefits|payments and other expenses "
        r"on employees|share based payments|"
        r"provident fund|pension|staff welfare|"
        r"non executive directors fees|compensation to employees",
        "Expenses",
        "Employee Costs"
    ),

    # FINANCIAL REVENUE
    (
        r"interest income|dividend income|dividend from group|"
        r"financial services income|fee based financial services income|"
        r"brokerage|commission fees|treasury operations|"
        r"gain on financial instruments|gain relating to forex|"
        r"profit on securitisation|share of profit|"
        r"financial instruments|royalty income",
        "Revenue",
        "Financial Revenue"
    ),

    # FINANCIAL EXPENSES
    (
        r"interest expense|interest on long term|interest on short term|"
        r"interest on finance lease|borrowing costs|"
        r"bank charges|bill discounting|"
        r"premium discount on debt|"
        r"financial services expenses|fund based financial services|"
        r"fee based financial services expenses|"
        r"guarantee fees",
        "Expenses",
        "Finance Costs"
    ),

    # OTHER FINANCIAL GAINS AND LOSSES
    (
        r"loss on securitisation|loss on financial instruments|"
        r"loss relating to forex|share of loss|"
        r"loss on sale of investment",
        "Gains and Losses",
        "Financial Gains and Losses"
    ),

    # TAX AND REGULATORY EXPENSES
    (
        r"indirect taxes|excise duty|sales tax|value added tax|"
        r"\bvat\b|service tax|tax penalties|penalties on taxes|"
        r"direct taxes|statutory levies|other indirect taxes|"
        r"duties|tax expenses",
        "Expenses",
        "Tax and Regulatory Costs"
    ),

    # CAPITALISATION AND DEDUCTIONS
    (
        r"capitalised|capitalized|transferred to dre|"
        r"income capitalised|interest capitalised",
        "Accounting Adjustments",
        "Capitalisation and Transfers"
    ),

    # NON-CASH EXPENSES
    (
        r"depreciation|amortisation|amortization|"
        r"impairment|provisions|credit balances written back",
        "Expenses",
        "Non-Cash Expenses"
    ),

    # ASSET AND INVESTMENT GAINS
    (
        r"gain on disposal|profit on sale of investment|"
        r"fair value gain|gain on step acquisition|"
        r"gain on restructuring|profit on corporate|"
        r"insurance claims|gain on sale of assets",
        "Gains and Losses",
        "Asset and Investment Gains"
    ),

    # OPERATING AND ADMINISTRATIVE COSTS
    (
        r"consultancy|auditors fees|audit fees|legal charges|"
        r"technical know how|technical service|royalt|"
        r"license fees|licence fees|rent lease|"
        r"insurance premium|advertising|marketing|"
        r"sales promotion|selling|distribution|"
        r"travel expenses|communications|telephone|"
        r"postage|courier|data centres|web hosting|"
        r"vsats|satellite links|isps|internet services|"
        r"software charges|it enabled services|"
        r"environment and pollution|research and development|"
        r"donations|community expenses|csr|"
        r"non executive directors fees|"
        r"other operational expenses|"
        r"miscellaneous expenditure|rebates|discount expenses",
        "Expenses",
        "Operating and Administrative Costs"
    ),

    # OPERATING REVENUE
    (
        r"manufacturing revenue|trading revenue|trading income|"
        r"sales of goods|job work income|construction income|"
        r"repairs maintenance|sale of electricity|"
        r"sale of gas|sale of water|"
        r"non financial services|export incentives|"
        r"fiscal benefits|other income|"
        r"rent income|operating lease rent income|"
        r"bad debts recovered|carbon credits",
        "Revenue",
        "Operating and Other Revenue"
    ),

    # ACCOUNTING ADJUSTMENTS
    (
        r"exceptional|one time debt waiver|"
        r"other adjustments|material exceptional income",
        "Accounting Adjustments",
        "Exceptional and Other Adjustments"
    )
]


# ------------------------------------------------------------
# 2. Improved classification function
# ------------------------------------------------------------

def refined_classification(variable):

    text = normalize_financial_text(variable)

    for pattern, domain, category in ONTOLOGY_RULES:

        if re.search(pattern, text):

            return pd.Series({
                "Refined_Domain": domain,
                "Refined_Category": category,
                "Matched_Rule": pattern,
                "Refined_Status": "Rule matched"
            })

    return pd.Series({
        "Refined_Domain": "Unclassified",
        "Refined_Category": "Unclassified",
        "Matched_Rule": None,
        "Refined_Status": "Requires review"
    })


# ------------------------------------------------------------
# 3. Apply refined ontology
# ------------------------------------------------------------

refined_results = ontology_df[
    "Original_Variable"
].apply(refined_classification)

refined_ontology_df = pd.concat(
    [
        ontology_df.reset_index(drop=True),
        refined_results.reset_index(drop=True)
    ],
    axis=1
)


# ------------------------------------------------------------
# 4. Correct accounting qualifier identification
# ------------------------------------------------------------

def refined_qualifier(text):

    text = str(text).lower().strip()

    if text.startswith("of which"):
        return "Subcomponent"

    if text.startswith("less"):
        return "Deduction"

    if "non-exceptional" in text or "non exceptional" in text:
        return "Non-exceptional"

    if "exceptional" in text:
        return "Exceptional"

    if "adjustment" in text:
        return "Adjustment"

    return "Primary"


refined_ontology_df["Refined_Qualifier"] = (
    refined_ontology_df["Original_Variable"]
    .apply(refined_qualifier)
)


# ------------------------------------------------------------
# 5. Classification improvement report
# ------------------------------------------------------------

old_matched = (
    ontology_df["Classification_Status"] == "Rule matched"
).sum()

new_matched = (
    refined_ontology_df["Refined_Status"] == "Rule matched"
).sum()

print("\nONTOLOGY IMPROVEMENT REPORT")
print("=" * 60)

print("Original matched:", old_matched)
print("Refined matched:", new_matched)

print("Newly classified:",
      new_matched - old_matched)

print("Original classification rate:",
      round(old_matched / len(ontology_df) * 100, 2), "%")

print("Refined classification rate:",
      round(new_matched / len(refined_ontology_df) * 100, 2), "%")


# ------------------------------------------------------------
# 6. Domain distribution
# ------------------------------------------------------------

print("\nREFINED FINANCIAL DOMAIN DISTRIBUTION")

display(
    refined_ontology_df[
        "Refined_Domain"
    ].value_counts().to_frame("Variable_Count")
)


# ------------------------------------------------------------
# 7. Category distribution
# ------------------------------------------------------------

print("\nREFINED FINANCIAL CATEGORY DISTRIBUTION")

category_report = (
    refined_ontology_df
    .groupby(
        ["Refined_Domain", "Refined_Category"]
    )
    .size()
    .reset_index(name="Variable_Count")
    .sort_values("Variable_Count", ascending=False)
)

display(category_report)


# ------------------------------------------------------------
# 8. Identify unresolved variables
# ------------------------------------------------------------

unresolved_variables = refined_ontology_df[
    refined_ontology_df["Refined_Status"] == "Requires review"
]

print("\nUNRESOLVED VARIABLES:",
      len(unresolved_variables))

display(
    unresolved_variables[
        [
            "Original_Variable",
            "Dataset",
            "Refined_Qualifier"
        ]
    ]
)


# ------------------------------------------------------------
# 9. Detect inconsistent duplicate labels
# ------------------------------------------------------------

duplicate_report = (
    refined_ontology_df
    .groupby("Normalized_Variable")
    .agg(
        Dataset_Count=("Dataset", "nunique"),
        Domain_Count=("Refined_Domain", "nunique"),
        Category_Count=("Refined_Category", "nunique")
    )
    .reset_index()
)

inconsistent_labels = duplicate_report[
    (duplicate_report["Dataset_Count"] > 1)
    & (
        (duplicate_report["Domain_Count"] > 1)
        | (duplicate_report["Category_Count"] > 1)
    )
]

print("\nINCONSISTENT DUPLICATE LABELS")
display(inconsistent_labels)


# ------------------------------------------------------------
# 10. Export refined ontology
# ------------------------------------------------------------

refined_ontology_df.to_csv(
    OUTPUT_DIR / "refined_financial_ontology.csv",
    index=False
)

category_report.to_csv(
    OUTPUT_DIR / "refined_financial_categories.csv",
    index=False
)

unresolved_variables.to_csv(
    OUTPUT_DIR / "unresolved_financial_variables.csv",
    index=False
)

print("\nRefined ontology saved successfully.")


ONTOLOGY IMPROVEMENT REPORT
Original matched: 129
Refined matched: 175
Newly classified: 46
Original classification rate: 70.88 %
Refined classification rate: 96.15 %

REFINED FINANCIAL DOMAIN DISTRIBUTION


,Variable_Count
Refined_Domain,
Expenses,82
Revenue,68
Gains and Losses,13
Unclassified,7
Accounting Adjustments,7
Inventory,5



REFINED FINANCIAL CATEGORY DISTRIBUTION


,Refined_Domain,Refined_Category,Variable_Count
6,Expenses,Operating and Administrative Costs,35
13,Revenue,Operating and Other Revenue,34
12,Revenue,Financial Revenue,32
3,Expenses,Finance Costs,14
2,Expenses,Employee Costs,11
9,Gains and Losses,Asset and Investment Gains,10
7,Expenses,Tax and Regulatory Costs,9
4,Expenses,Material and Production Costs,7
15,Unclassified,Unclassified,7
11,Inventory,Inventory Changes,5



UNRESOLVED VARIABLES: 7


,Original_Variable,Dataset,Refined_Qualifier
14,Income from financial services,PL_Composition,Primary
69,Income from financial services,Change_Expenses,Primary
72,Water charges,Change_Expenses,Primary
103,Less: indirect tax credits,Change_Expenses,Deduction
127,Outsourced professional services,Change_Expenses,Primary
135,Other professional services,Change_Expenses,Primary
159,Research & development expenses,Change_Expenses,Primary



INCONSISTENT DUPLICATE LABELS


,Normalized_Variable,Dataset_Count,Domain_Count,Category_Count



Refined ontology saved successfully.


In [9]:
# ============================================================
# PART 6: FINALIZE FINANCIAL ONTOLOGY
# ============================================================

import hashlib
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Resolve remaining variables
# ------------------------------------------------------------

MANUAL_OVERRIDES = {

    "income from financial services": (
        "Revenue", "Financial Revenue"
    ),

    "water charges": (
        "Expenses", "Operating and Administrative Costs"
    ),

    "less indirect tax credits": (
        "Accounting Adjustments", "Tax Credits and Deductions"
    ),

    "outsourced professional services": (
        "Expenses", "Operating and Administrative Costs"
    ),

    "other professional services": (
        "Expenses", "Operating and Administrative Costs"
    ),

    "research development expenses": (
        "Expenses", "Operating and Administrative Costs"
    )
}


def apply_manual_overrides(row):

    key = row["Normalized_Variable"]

    if key in MANUAL_OVERRIDES:

        domain, category = MANUAL_OVERRIDES[key]

        row["Refined_Domain"] = domain
        row["Refined_Category"] = category
        row["Refined_Status"] = "Manual accounting review"
        row["Matched_Rule"] = "Manual override"

    return row


refined_ontology_df = refined_ontology_df.apply(
    apply_manual_overrides,
    axis=1
)


# ------------------------------------------------------------
# 2. Create stable concept identifiers
# ------------------------------------------------------------

def generate_concept_id(text):

    normalized = normalize_financial_text(text)

    return "FIN_" + hashlib.sha256(
        normalized.encode("utf-8")
    ).hexdigest()[:12].upper()


refined_ontology_df["Concept_ID"] = (
    refined_ontology_df["Normalized_Variable"]
    .apply(generate_concept_id)
)


# ------------------------------------------------------------
# 3. Create hierarchical identifiers
# ------------------------------------------------------------

refined_ontology_df["Domain_ID"] = (
    refined_ontology_df["Refined_Domain"]
    .apply(generate_concept_id)
)

refined_ontology_df["Category_ID"] = (
    refined_ontology_df["Refined_Domain"]
    + "::"
    + refined_ontology_df["Refined_Category"]
).apply(generate_concept_id)


# ------------------------------------------------------------
# 4. Generate semantic descriptions
# ------------------------------------------------------------

CATEGORY_DESCRIPTIONS = {

    "Total Revenue":
        "Aggregate income reported from continued operations.",

    "Total Expenses":
        "Aggregate expenses reported for the company's operations.",

    "Operating Revenue":
        "Revenue associated with non-financial business activities.",

    "Operating and Other Revenue":
        "Income from operating activities and other reported revenue sources.",

    "Financial Revenue":
        "Income generated from financial services and financial activities.",

    "Other Revenue":
        "Other reported income and revenue-related items.",

    "Material and Production Costs":
        "Costs associated with materials, production inputs and manufacturing.",

    "Employee Costs":
        "Expenses associated with employee compensation and benefits.",

    "Operating and Administrative Costs":
        "Expenses associated with administration, operations and business support.",

    "Finance Costs":
        "Expenses associated with financing, borrowing and financial services.",

    "Non-Cash Expenses":
        "Non-cash accounting expenses including depreciation and impairment.",

    "Tax and Regulatory Costs":
        "Tax-related expenses, statutory duties and regulatory charges.",

    "Inventory Changes":
        "Changes in inventory, finished goods and work in progress.",

    "Asset and Investment Gains":
        "Gains associated with assets, investments and valuation changes.",

    "Financial Gains and Losses":
        "Gains and losses associated with financial instruments and transactions.",

    "Capitalisation and Transfers":
        "Financial amounts capitalised or transferred between accounting categories.",

    "Exceptional and Other Adjustments":
        "Exceptional financial items and other accounting adjustments.",

    "Tax Credits and Deductions":
        "Tax-related credits and deductions reported in financial statements."
}


def create_semantic_description(row):

    category = row["Refined_Category"]

    description = CATEGORY_DESCRIPTIONS.get(
        category,
        "Financial statement variable requiring semantic review."
    )

    return (
        f"Financial concept: {row['Normalized_Variable']}. "
        f"Financial domain: {row['Refined_Domain']}. "
        f"Financial category: {category}. "
        f"Accounting context: {row['Refined_Qualifier']}. "
        f"{description}"
    )


refined_ontology_df["Semantic_Description"] = (
    refined_ontology_df.apply(
        create_semantic_description,
        axis=1
    )
)


# ------------------------------------------------------------
# 5. Create hierarchy paths
# ------------------------------------------------------------

refined_ontology_df["Ontology_Path"] = (
    refined_ontology_df["Refined_Domain"]
    + " > "
    + refined_ontology_df["Refined_Category"]
    + " > "
    + refined_ontology_df["Normalized_Variable"]
)


# ------------------------------------------------------------
# 6. Build final NLP dictionary
# ------------------------------------------------------------

final_columns = [

    "Concept_ID",
    "Original_Variable",
    "Normalized_Variable",
    "Dataset",
    "Refined_Domain",
    "Refined_Category",
    "Refined_Qualifier",
    "Domain_ID",
    "Category_ID",
    "Ontology_Path",
    "Semantic_Description",
    "Refined_Status"
]

financial_dictionary = (
    refined_ontology_df[final_columns]
    .copy()
)


# ------------------------------------------------------------
# 7. Build unique concept table
# ------------------------------------------------------------

concept_table = (
    financial_dictionary
    .groupby("Concept_ID")
    .agg(
        Financial_Variable=("Normalized_Variable", "first"),
        Financial_Domain=("Refined_Domain", "first"),
        Financial_Category=("Refined_Category", "first"),
        Semantic_Description=("Semantic_Description", "first"),
        Dataset_Count=("Dataset", "nunique"),
        Dataset_Origin=(
            "Dataset",
            lambda x: " | ".join(sorted(set(x)))
        )
    )
    .reset_index()
)

print("Total variable records:", len(financial_dictionary))
print("Unique financial concepts:", len(concept_table))

print("\nClassification status:")
display(
    financial_dictionary["Refined_Status"]
    .value_counts()
)

print("\nFinal ontology:")
display(financial_dictionary.head(20))

print("\nUnique concept table:")
display(concept_table.head(20))


# ------------------------------------------------------------
# 8. Validation
# ------------------------------------------------------------

assert len(financial_dictionary) == 182

assert financial_dictionary["Concept_ID"].notna().all()

assert financial_dictionary["Refined_Domain"].ne(
    "Unclassified"
).all(), "Some variables remain unclassified."

assert financial_dictionary["Concept_ID"].nunique() == 141

print("\nAll ontology validation checks passed.")


# ------------------------------------------------------------
# 9. Export
# ------------------------------------------------------------

financial_dictionary.to_csv(
    OUTPUT_DIR / "final_financial_nlp_dictionary.csv",
    index=False
)

concept_table.to_csv(
    OUTPUT_DIR / "unique_financial_concepts.csv",
    index=False
)

print("\nExport completed:")
print("1. final_financial_nlp_dictionary.csv")
print("2. unique_financial_concepts.csv")

Total variable records: 182
Unique financial concepts: 141

Classification status:


Refined_Status
Rule matched                175
Manual accounting review      7
Name: count, dtype: int64


Final ontology:


,Concept_ID,Original_Variable,Normalized_Variable,Dataset,Refined_Domain,Refined_Category,Refined_Qualifier,Domain_ID,Category_ID,Ontology_Path,Semantic_Description,Refined_Status
0,FIN_A5D21E48CFE5,Total Income from continued operations,total income from continued operations,PL_Composition,Revenue,Total Revenue,Primary,FIN_371F7E422380,FIN_85C9144F5C9C,Revenue > Total Revenue > total income from co...,Financial concept: total income from continued...,Rule matched
1,FIN_BEB60B8338C0,Income from non-financial services,income from non financial services,PL_Composition,Revenue,Operating and Other Revenue,Primary,FIN_371F7E422380,FIN_337DB2F811D9,Revenue > Operating and Other Revenue > income...,Financial concept: income from non financial s...,Rule matched
2,FIN_958A75FAB61F,Manufacturing revenue,manufacturing revenue,PL_Composition,Revenue,Operating and Other Revenue,Primary,FIN_371F7E422380,FIN_337DB2F811D9,Revenue > Operating and Other Revenue > manufa...,Financial concept: manufacturing revenue. Fina...,Rule matched
3,FIN_3819CC19E4FC,Trading revenue and non-financial services,trading revenue and non financial services,PL_Composition,Revenue,Operating and Other Revenue,Primary,FIN_371F7E422380,FIN_337DB2F811D9,Revenue > Operating and Other Revenue > tradin...,Financial concept: trading revenue and non fin...,Rule matched
4,FIN_3C17F9F16265,Sales of goods,sales of goods,PL_Composition,Revenue,Operating and Other Revenue,Primary,FIN_371F7E422380,FIN_337DB2F811D9,Revenue > Operating and Other Revenue > sales ...,Financial concept: sales of goods. Financial d...,Rule matched
5,FIN_D2CC1B5E7D9A,Job-work income,job work income,PL_Composition,Revenue,Operating and Other Revenue,Primary,FIN_371F7E422380,FIN_337DB2F811D9,Revenue > Operating and Other Revenue > job wo...,Financial concept: job work income. Financial ...,Rule matched
6,FIN_D283F080F0A7,Income from repairs & maintenance including af...,income from repairs maintenance including afte...,PL_Composition,Revenue,Operating and Other Revenue,Primary,FIN_371F7E422380,FIN_337DB2F811D9,Revenue > Operating and Other Revenue > income...,Financial concept: income from repairs mainten...,Rule matched
7,FIN_888CA5D5C2F1,Construction income,construction income,PL_Composition,Revenue,Operating and Other Revenue,Primary,FIN_371F7E422380,FIN_337DB2F811D9,Revenue > Operating and Other Revenue > constr...,Financial concept: construction income. Financ...,Rule matched
8,FIN_E59E95731018,"Sale of electricity, gas and water",sale of electricity gas and water,PL_Composition,Revenue,Operating and Other Revenue,Primary,FIN_371F7E422380,FIN_337DB2F811D9,Revenue > Operating and Other Revenue > sale o...,Financial concept: sale of electricity gas and...,Rule matched
9,FIN_1CC010643954,Fiscal benefits,fiscal benefits,PL_Composition,Revenue,Operating and Other Revenue,Primary,FIN_371F7E422380,FIN_337DB2F811D9,Revenue > Operating and Other Revenue > fiscal...,Financial concept: fiscal benefits. Financial ...,Rule matched



Unique concept table:


,Concept_ID,Financial_Variable,Financial_Domain,Financial_Category,Semantic_Description,Dataset_Count,Dataset_Origin
0,FIN_03072DDDEA19,interest on short term borrowings bank overdra...,Expenses,Finance Costs,Financial concept: interest on short term borr...,1,Change_Expenses
1,FIN_0853F25AD750,gain on disposal of non current non financial ...,Gains and Losses,Asset and Investment Gains,Financial concept: gain on disposal of non cur...,2,Change_Expenses | PL_Composition
2,FIN_0934223437CE,technical know how fees and technical service ...,Expenses,Operating and Administrative Costs,Financial concept: technical know how fees and...,1,Change_Expenses
3,FIN_0B56A17EEA53,postage courier,Expenses,Operating and Administrative Costs,Financial concept: postage courier. Financial ...,1,Change_Expenses
4,FIN_0BD8ED8EC598,interest expense,Expenses,Finance Costs,Financial concept: interest expense. Financial...,1,Change_Expenses
5,FIN_12AD529248EC,social and community expenses including csr exp,Expenses,Operating and Administrative Costs,Financial concept: social and community expens...,1,Change_Expenses
6,FIN_17347058195F,consultancy fees to auditors,Expenses,Operating and Administrative Costs,Financial concept: consultancy fees to auditor...,1,Change_Expenses
7,FIN_188E881C7BB3,water charges,Expenses,Operating and Administrative Costs,Financial concept: water charges. Financial do...,1,Change_Expenses
8,FIN_18A52D241144,purchase of finished goods,Inventory,Inventory Changes,Financial concept: purchase of finished goods....,1,Change_Expenses
9,FIN_1B1E84C7DB4D,expenses on data centres web hosting and co ho...,Expenses,Operating and Administrative Costs,Financial concept: expenses on data centres we...,1,Change_Expenses



All ontology validation checks passed.

Export completed:
1. final_financial_nlp_dictionary.csv
2. unique_financial_concepts.csv


In [10]:

# CELL 9: Prepare unique financial concepts for NLP embeddings

import pandas as pd

# Keep one record per unique financial concept
embedding_corpus = (
    financial_dictionary
    .sort_values(["Concept_ID", "Dataset"])
    .drop_duplicates(subset="Concept_ID")
    .copy()
)

# Representation 1: Original financial terminology
embedding_corpus["Raw_Text"] = (
    embedding_corpus["Original_Variable"].astype(str)
)

# Representation 2: Normalized financial terminology
embedding_corpus["Normalized_Text"] = (
    embedding_corpus["Normalized_Variable"].astype(str)
)

# Representation 3: Ontology-enriched terminology
embedding_corpus["Ontology_Enriched_Text"] = (
    "Financial concept: "
    + embedding_corpus["Normalized_Variable"].astype(str)
    + ". Domain: "
    + embedding_corpus["Refined_Domain"].astype(str)
    + ". Category: "
    + embedding_corpus["Refined_Category"].astype(str)
    + ". Accounting qualifier: "
    + embedding_corpus["Refined_Qualifier"].astype(str)
    + ". Description: "
    + embedding_corpus["Semantic_Description"].astype(str)
)

# Select columns required for NLP experiments
embedding_corpus = embedding_corpus[
    [
        "Concept_ID",
        "Original_Variable",
        "Normalized_Variable",
        "Refined_Domain",
        "Refined_Category",
        "Refined_Qualifier",
        "Raw_Text",
        "Normalized_Text",
        "Ontology_Enriched_Text"
    ]
].reset_index(drop=True)

# Validation
assert embedding_corpus["Concept_ID"].nunique() == 141
assert len(embedding_corpus) == 141
assert embedding_corpus["Ontology_Enriched_Text"].notna().all()

print("Unique concepts prepared:", len(embedding_corpus))
print("Unique Concept IDs:", embedding_corpus["Concept_ID"].nunique())

print("\nText representation availability:")
print("Raw labels:", embedding_corpus["Raw_Text"].notna().sum())
print("Normalized labels:", embedding_corpus["Normalized_Text"].notna().sum())
print("Ontology-enriched labels:", embedding_corpus["Ontology_Enriched_Text"].notna().sum())

print("\nSample NLP corpus:")
display(
    embedding_corpus[
        [
            "Concept_ID",
            "Raw_Text",
            "Normalized_Text",
            "Ontology_Enriched_Text"
        ]
    ].head(10)
)

# Save the NLP corpus
embedding_corpus.to_csv(
    OUTPUT_DIR / "financial_nlp_embedding_corpus.csv",
    index=False
)

print("\nSaved: financial_nlp_embedding_corpus.csv")

Unique concepts prepared: 141
Unique Concept IDs: 141

Text representation availability:
Raw labels: 141
Normalized labels: 141
Ontology-enriched labels: 141

Sample NLP corpus:


,Concept_ID,Raw_Text,Normalized_Text,Ontology_Enriched_Text
0,FIN_03072DDDEA19,Interest on short term borrowings / bank overd...,interest on short term borrowings bank overdra...,Financial concept: interest on short term borr...
1,FIN_0853F25AD750,Gain on disposal of non-current non-financial ...,gain on disposal of non current non financial ...,Financial concept: gain on disposal of non cur...
2,FIN_0934223437CE,Technical know-how fees and technical service ...,technical know how fees and technical service ...,Financial concept: technical know how fees and...
3,FIN_0B56A17EEA53,Postage & courier,postage courier,Financial concept: postage courier. Domain: Ex...
4,FIN_0BD8ED8EC598,Interest expense,interest expense,Financial concept: interest expense. Domain: E...
5,FIN_12AD529248EC,Social and community expenses (including CSR exp),social and community expenses including csr exp,Financial concept: social and community expens...
6,FIN_17347058195F,Consultancy fees to auditors,consultancy fees to auditors,Financial concept: consultancy fees to auditor...
7,FIN_188E881C7BB3,Water charges,water charges,Financial concept: water charges. Domain: Expe...
8,FIN_18A52D241144,Purchase of finished goods,purchase of finished goods,Financial concept: purchase of finished goods....
9,FIN_1B1E84C7DB4D,"Expenses on data centres, web hosting and co h...",expenses on data centres web hosting and co ho...,Financial concept: expenses on data centres we...



Saved: financial_nlp_embedding_corpus.csv


In [11]:

# CELL 10: Install and import NLP libraries

%pip install -q sentence-transformers transformers scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
# CELL 10B: Import NLP libraries and verify environment

import numpy as np
import pandas as pd
import torch

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModel

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

print("\nNLP libraries imported successfully.")

PyTorch version: 2.14.0+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU

NLP libraries imported successfully.


In [13]:
# CELL 11: Generate TF-IDF embeddings

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Initialize separate vectorizers for fair comparison
tfidf_vectorizers = {
    "raw": TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=1,
        sublinear_tf=True
    ),
    "normalized": TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=1,
        sublinear_tf=True
    ),
    "ontology": TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=1,
        sublinear_tf=True
    )
}

# Generate TF-IDF matrices
tfidf_embeddings = {}

text_columns = {
    "raw": "Raw_Text",
    "normalized": "Normalized_Text",
    "ontology": "Ontology_Enriched_Text"
}

for representation, column in text_columns.items():

    texts = embedding_corpus[column].fillna("").tolist()

    matrix = tfidf_vectorizers[representation].fit_transform(texts)

    tfidf_embeddings[representation] = matrix

    print(f"\n{representation.upper()} representation")
    print("Embedding shape:", matrix.shape)
    print("Vocabulary size:", len(tfidf_vectorizers[representation].vocabulary_))

# Validate
assert all(
    matrix.shape[0] == len(embedding_corpus)
    for matrix in tfidf_embeddings.values()
)

print("\nAll TF-IDF embeddings generated successfully.")


RAW representation
Embedding shape: (141, 639)
Vocabulary size: 639

NORMALIZED representation
Embedding shape: (141, 639)
Vocabulary size: 639

ONTOLOGY representation
Embedding shape: (141, 1090)
Vocabulary size: 1090

All TF-IDF embeddings generated successfully.


In [14]:
# CELL 12: Compare nearest financial concepts

def show_nearest_concepts(concept_query, representation="raw", top_n=5):

    if representation not in tfidf_embeddings:
        raise ValueError("Representation must be raw, normalized, or ontology.")

    # Find the requested concept
    matches = embedding_corpus[
        embedding_corpus["Normalized_Variable"].str.contains(
            concept_query, case=False, regex=False, na=False
        )
    ]

    if matches.empty:
        print("No matching financial concept found.")
        return

    query_idx = matches.index[0]

    # Compute cosine similarity against all concepts
    similarity_matrix = cosine_similarity(
        tfidf_embeddings[representation]
    )

    scores = similarity_matrix[query_idx]

    results = embedding_corpus[
        [
            "Concept_ID",
            "Original_Variable",
            "Refined_Domain",
            "Refined_Category"
        ]
    ].copy()

    results["Similarity"] = scores

    # Exclude the query concept itself
    results = results.drop(index=query_idx)

    results = results.sort_values(
        "Similarity", ascending=False
    ).head(top_n)

    print("Query concept:", matches.iloc[0]["Original_Variable"])
    print("Representation:", representation.upper())

    display(results.reset_index(drop=True))


# Test selected financial concepts
test_concepts = [
    "interest income",
    "employee benefits",
    "depreciation",
    "sales of goods",
    "finance costs"
]

for concept in test_concepts:

    print("\n" + "=" * 80)

    show_nearest_concepts(
        concept_query=concept,
        representation="raw",
        top_n=5
    )


Query concept: Interest income from banks
Representation: RAW


,Concept_ID,Original_Variable,Refined_Domain,Refined_Category,Similarity
0,FIN_E3EF503E4328,Interest income,Revenue,Financial Revenue,0.58
1,FIN_FA909C160A84,Interest income of cos other than banks from i...,Revenue,Financial Revenue,0.37
2,FIN_8523C43B68BA,Income from financial services,Revenue,Financial Revenue,0.34
3,FIN_8A0045F3B426,Interest income of cos other than banks from l...,Revenue,Financial Revenue,0.33
4,FIN_FAF037CFF47C,Income from treasury operations,Revenue,Financial Revenue,0.29



Query concept: Employee benefits expenses
Representation: RAW


,Concept_ID,Original_Variable,Refined_Domain,Refined_Category,Similarity
0,FIN_1CC010643954,Fiscal benefits,Revenue,Operating and Other Revenue,0.25
1,FIN_FF18FCEBF80B,Financial services expenses,Expenses,Finance Costs,0.08
2,FIN_7BFCB554EDCA,Communications expenses,Expenses,Operating and Administrative Costs,0.08
3,FIN_9B2B7504FCDA,Telephone expenses,Expenses,Operating and Administrative Costs,0.08
4,FIN_9E9AE3D98900,Marketing expenses,Expenses,Operating and Administrative Costs,0.08



Query concept: Expenses other than Depreciation, Interest, Taxes, Provisions and Amortizations
Representation: RAW


,Concept_ID,Original_Variable,Refined_Domain,Refined_Category,Similarity
0,FIN_FA909C160A84,Interest income of cos other than banks from i...,Revenue,Financial Revenue,0.19
1,FIN_8A0045F3B426,Interest income of cos other than banks from l...,Revenue,Financial Revenue,0.18
2,FIN_FDE0DEFF0020,Other indirect taxes,Expenses,Tax and Regulatory Costs,0.14
3,FIN_393FE81F02FB,Indirect taxes,Expenses,Tax and Regulatory Costs,0.11
4,FIN_E3EF503E4328,Interest income,Revenue,Financial Revenue,0.10



Query concept: Sales of goods
Representation: RAW


,Concept_ID,Original_Variable,Refined_Domain,Refined_Category,Similarity
0,FIN_35CE64B616E4,Sales tax,Expenses,Tax and Regulatory Costs,0.24
1,FIN_18A52D241144,Purchase of finished goods,Inventory,Inventory Changes,0.20
2,FIN_380106D57CCF,Sales promotion expenses,Expenses,Operating and Administrative Costs,0.18
3,FIN_5CAF3B5F43E5,Goods and service tax,Expenses,Tax and Regulatory Costs,0.15
4,FIN_D975467FA329,Change in stock of finished goods (including i...,Inventory,Inventory Changes,0.13



No matching financial concept found.


In [15]:
# CELL 13: Generate Sentence-BERT embeddings

import numpy as np
import torch
from sentence_transformers import SentenceTransformer

# Select GPU when available
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load pretrained Sentence-BERT model
sbert_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=device
)

# Generate embeddings for three representations
sbert_embeddings = {}

sbert_text_columns = {
    "raw": "Raw_Text",
    "normalized": "Normalized_Text",
    "ontology": "Ontology_Enriched_Text"
}

for representation, column in sbert_text_columns.items():

    texts = embedding_corpus[column].fillna("").tolist()

    embeddings = sbert_model.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    sbert_embeddings[representation] = embeddings

    print(f"\n{representation.upper()} representation")
    print("Embedding shape:", embeddings.shape)
    print("Embedding dimension:", embeddings.shape[1])

# Validation
assert sbert_embeddings["raw"].shape == (141, 384)
assert sbert_embeddings["normalized"].shape == (141, 384)
assert sbert_embeddings["ontology"].shape == (141, 384)

# Save embeddings
for representation, embeddings in sbert_embeddings.items():

    np.save(
        OUTPUT_DIR / f"sbert_{representation}_embeddings.npy",
        embeddings
    )

print("\nAll Sentence-BERT embeddings generated successfully.")
print("Device used:", device)

Batches: 100%|██████████| 5/5 [00:00<00:00,  6.75it/s]



RAW representation
Embedding shape: (141, 384)
Embedding dimension: 384


Batches: 100%|██████████| 5/5 [00:00<00:00, 82.83it/s]



NORMALIZED representation
Embedding shape: (141, 384)
Embedding dimension: 384


Batches: 100%|██████████| 5/5 [00:00<00:00, 62.31it/s]


ONTOLOGY representation
Embedding shape: (141, 384)
Embedding dimension: 384

All Sentence-BERT embeddings generated successfully.
Device used: cuda


In [16]:
# CELL 14: Sentence-BERT nearest-neighbour analysis

from sklearn.metrics.pairwise import cosine_similarity

def show_sbert_nearest_concepts(
    concept_query,
    representation="raw",
    top_n=5
):

    if representation not in sbert_embeddings:
        raise ValueError("Choose raw, normalized, or ontology.")

    matches = embedding_corpus[
        embedding_corpus["Normalized_Variable"].str.contains(
            concept_query,
            case=False,
            regex=False,
            na=False
        )
    ]

    if matches.empty:
        print("No matching financial concept found:", concept_query)
        return

    query_idx = matches.index[0]

    # Calculate semantic similarities
    scores = cosine_similarity(
        sbert_embeddings[representation][query_idx].reshape(1, -1),
        sbert_embeddings[representation]
    )[0]

    results = embedding_corpus[
        [
            "Concept_ID",
            "Original_Variable",
            "Refined_Domain",
            "Refined_Category"
        ]
    ].copy()

    results["Similarity"] = scores

    # Remove query concept itself
    results = results.drop(index=query_idx)

    results = results.sort_values(
        "Similarity",
        ascending=False
    ).head(top_n)

    print("Query:", matches.iloc[0]["Original_Variable"])
    print("Representation:", representation.upper())

    display(results.reset_index(drop=True))


# Test the same financial concepts
test_concepts = [
    "interest income",
    "employee benefits",
    "depreciation",
    "sales of goods",
    "finance costs"
]

for concept in test_concepts:

    print("\n" + "=" * 80)

    show_sbert_nearest_concepts(
        concept_query=concept,
        representation="raw",
        top_n=5
    )


Query: Interest income from banks
Representation: RAW


,Concept_ID,Original_Variable,Refined_Domain,Refined_Category,Similarity
0,FIN_E3EF503E4328,Interest income,Revenue,Financial Revenue,0.75
1,FIN_FA909C160A84,Interest income of cos other than banks from i...,Revenue,Financial Revenue,0.72
2,FIN_8A0045F3B426,Interest income of cos other than banks from l...,Revenue,Financial Revenue,0.67
3,FIN_8523C43B68BA,Income from financial services,Revenue,Financial Revenue,0.59
4,FIN_0BD8ED8EC598,Interest expense,Expenses,Finance Costs,0.57



Query: Employee benefits expenses
Representation: RAW


,Concept_ID,Original_Variable,Refined_Domain,Refined_Category,Similarity
0,FIN_85132C1B0D2F,Payments and other expenses on employees,Expenses,Employee Costs,0.74
1,FIN_3D54BB08B396,Staff welfare & training expenses,Expenses,Employee Costs,0.60
2,FIN_FF18FCEBF80B,Financial services expenses,Expenses,Finance Costs,0.53
3,FIN_9FFC7914BF58,Total expenses excluding direct taxes,Expenses,Total Expenses,0.49
4,FIN_1CC010643954,Fiscal benefits,Revenue,Operating and Other Revenue,0.48



Query: Expenses other than Depreciation, Interest, Taxes, Provisions and Amortizations
Representation: RAW


,Concept_ID,Original_Variable,Refined_Domain,Refined_Category,Similarity
0,FIN_78A126F37B09,Other interest expense (term not specified),Expenses,Finance Costs,0.64
1,FIN_FF18FCEBF80B,Financial services expenses,Expenses,Finance Costs,0.63
2,FIN_54A50EA6CE42,Ordinary and non-exceptional expenses,Expenses,Total Expenses,0.63
3,FIN_0BD8ED8EC598,Interest expense,Expenses,Finance Costs,0.62
4,FIN_903FDBA79BF6,Selling & distribution expenses,Expenses,Operating and Administrative Costs,0.60



Query: Sales of goods
Representation: RAW


,Concept_ID,Original_Variable,Refined_Domain,Refined_Category,Similarity
0,FIN_18A52D241144,Purchase of finished goods,Inventory,Inventory Changes,0.73
1,FIN_958A75FAB61F,Manufacturing revenue,Revenue,Operating and Other Revenue,0.62
2,FIN_35CE64B616E4,Sales tax,Expenses,Tax and Regulatory Costs,0.56
3,FIN_5CAF3B5F43E5,Goods and service tax,Expenses,Tax and Regulatory Costs,0.54
4,FIN_B0B1FC31CB09,"Stores, spares, tools consumed",Expenses,Material and Production Costs,0.50



No matching financial concept found: finance costs


In [ ]:
# CELL 15: Compare SBERT representations quantitatively

from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np

REPRESENTATIONS = ["raw", "normalized", "ontology"]

def evaluate_neighbours(query, representation, top_n=5):

    matches = embedding_corpus[
        embedding_corpus["Normalized_Variable"].str.contains(
            query,
            case=False,
            regex=False,
            na=False
        )
    ]

    if matches.empty:
        return None

    query_idx = matches.index[0]

    similarities = cosine_similarity(
        sbert_embeddings[representation][query_idx].reshape(1, -1),
        sbert_embeddings[representation]
    )[0]

    results = embedding_corpus.copy()
    results["Similarity"] = similarities

    results = results.drop(index=query_idx)

    results = results.sort_values(
        "Similarity",
        ascending=False
    ).head(top_n)

    query_category = embedding_corpus.loc[
        query_idx, "Refined_Category"
    ]

    query_domain = embedding_corpus.loc[
        query_idx, "Refined_Domain"
    ]

    category_consistency = (
        results["Refined_Category"] == query_category
    ).mean()

    domain_consistency = (
        results["Refined_Domain"] == query_domain
    ).mean()

    return {
        "Query": embedding_corpus.loc[
            query_idx, "Original_Variable"
        ],
        "Representation": representation,
        "Mean_Similarity": results["Similarity"].mean(),
        "Category_Consistency": category_consistency,
        "Domain_Consistency": domain_consistency,
        "Nearest_Neighbours": results[
            [
                "Original_Variable",
                "Refined_Domain",
                "Refined_Category",
                "Similarity"
            ]
        ].reset_index(drop=True)
    }


# Use representative concepts that exist in the corpus
queries = [
    "interest income from banks",
    "employee benefits expenses",
    "sales of goods",
    "depreciation"
]

comparison_records = []

for query in queries:

    for representation in REPRESENTATIONS:

        result = evaluate_neighbours(
            query,
            representation,
            top_n=5
        )

        if result is not None:

            comparison_records.append({
                "Query": result["Query"],
                "Representation": representation,
                "Mean_Similarity": result["Mean_Similarity"],
                "Category_Consistency": result["Category_Consistency"],
                "Domain_Consistency": result["Domain_Consistency"]
            })

comparison_df = pd.DataFrame(comparison_records)

display(
    comparison_df.round(4)
)

print("\nRepresentation-level averages:")

display(
    comparison_df.groupby("Representation")[
        [
            "Mean_Similarity",
            "Category_Consistency",
            "Domain_Consistency"
        ]
    ].mean().round(4)
)

,Query,Representation,Mean_Similarity,Category_Consistency,Domain_Consistency
0,Interest income from banks,raw,0.66,0.80,0.80
1,Interest income from banks,normalized,0.66,0.80,0.80
2,Interest income from banks,ontology,0.94,1.00,1.00
3,Employee benefits expenses,raw,0.57,0.40,0.80
4,Employee benefits expenses,normalized,0.56,0.40,0.80
5,Employee benefits expenses,ontology,0.89,0.80,1.00
6,Sales of goods,raw,0.59,0.20,0.20
7,Sales of goods,normalized,0.59,0.20,0.20
8,Sales of goods,ontology,0.94,1.00,1.00
9,"Expenses other than Depreciation, Interest, Ta...",raw,0.63,0.00,1.00



Representation-level averages:


,Mean_Similarity,Category_Consistency,Domain_Consistency
Representation,,,
normalized,0.60,0.35,0.70
ontology,0.91,0.75,1.00
raw,0.61,0.35,0.70


In [18]:
# CELL 16: Measure ontology-induced similarity changes

from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np

# Calculate pairwise similarity matrices
raw_similarity = cosine_similarity(
    sbert_embeddings["raw"]
)

ontology_similarity = cosine_similarity(
    sbert_embeddings["ontology"]
)

# Define selected concept pairs for inspection
concept_pairs = [
    ("interest income from banks", "interest income"),
    ("employee benefits expenses", "staff welfare"),
    ("sales of goods", "purchase of finished goods"),
    ("depreciation", "amortization"),
    ("interest income", "interest expense")
]

pair_results = []

for concept_a, concept_b in concept_pairs:

    match_a = embedding_corpus[
        embedding_corpus["Normalized_Variable"].str.contains(
            concept_a, case=False, regex=False, na=False
        )
    ]

    match_b = embedding_corpus[
        embedding_corpus["Normalized_Variable"].str.contains(
            concept_b, case=False, regex=False, na=False
        )
    ]

    if match_a.empty or match_b.empty:
        print(f"Skipping unmatched pair: {concept_a} / {concept_b}")
        continue

    idx_a = match_a.index[0]
    idx_b = match_b.index[0]

    raw_score = raw_similarity[idx_a, idx_b]
    ontology_score = ontology_similarity[idx_a, idx_b]

    pair_results.append({
        "Concept_A": embedding_corpus.loc[
            idx_a, "Original_Variable"
        ],
        "Concept_B": embedding_corpus.loc[
            idx_b, "Original_Variable"
        ],
        "Raw_Similarity": raw_score,
        "Ontology_Similarity": ontology_score,
        "Similarity_Change": ontology_score - raw_score,
        "Same_Category": (
            embedding_corpus.loc[idx_a, "Refined_Category"]
            == embedding_corpus.loc[idx_b, "Refined_Category"]
        )
    })

pair_comparison_df = pd.DataFrame(pair_results)

display(pair_comparison_df.round(4))

print("\nAverage similarity change:")
print(
    pair_comparison_df["Similarity_Change"].mean().round(4)
)


,Concept_A,Concept_B,Raw_Similarity,Ontology_Similarity,Similarity_Change,Same_Category
0,Interest income from banks,Interest income from banks,1.00,1.00,-0.00,True
1,Employee benefits expenses,Staff welfare & training expenses,0.60,0.86,0.26,True
2,Sales of goods,Purchase of finished goods,0.73,0.83,0.09,False
3,"Expenses other than Depreciation, Interest, Ta...","Expenses other than Depreciation, Interest, Ta...",1.00,1.00,-0.00,True
4,Interest income from banks,Interest expense,0.57,0.87,0.30,False



Average similarity change:
0.1307


In [19]:
# CELL 17: Generate FinBERT contextual embeddings

import torch
import torch.nn.functional as F
import numpy as np
from transformers import AutoTokenizer, AutoModel

# Select device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load finance-domain pretrained model
FINBERT_NAME = "ProsusAI/finbert"

finbert_tokenizer = AutoTokenizer.from_pretrained(FINBERT_NAME)

finbert_model = AutoModel.from_pretrained(
    FINBERT_NAME
).to(device)

finbert_model.eval()

print("FinBERT loaded successfully.")
print("Device:", device)

# Mean pooling with attention mask
def mean_pooling(model_output, attention_mask):

    token_embeddings = model_output.last_hidden_state

    mask = attention_mask.unsqueeze(-1).expand(
        token_embeddings.size()
    ).float()

    summed_embeddings = torch.sum(
        token_embeddings * mask, dim=1
    )

    token_counts = torch.clamp(
        mask.sum(dim=1), min=1e-9
    )

    return summed_embeddings / token_counts


# Generate embeddings for each text representation
finbert_embeddings = {}

finbert_text_columns = {
    "raw": "Raw_Text",
    "normalized": "Normalized_Text",
    "ontology": "Ontology_Enriched_Text"
}

for representation, column in finbert_text_columns.items():

    texts = embedding_corpus[column].fillna("").tolist()

    all_embeddings = []

    for start in range(0, len(texts), 16):

        batch_texts = texts[start:start + 16]

        encoded = finbert_tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )

        encoded = {
            key: value.to(device)
            for key, value in encoded.items()
        }

        with torch.inference_mode():

            outputs = finbert_model(**encoded)

            pooled = mean_pooling(
                outputs,
                encoded["attention_mask"]
            )

            pooled = F.normalize(pooled, p=2, dim=1)

        all_embeddings.append(
            pooled.cpu().numpy()
        )

    embeddings = np.concatenate(all_embeddings, axis=0)

    finbert_embeddings[representation] = embeddings

    print(f"\n{representation.upper()} representation")
    print("Embedding shape:", embeddings.shape)
    print("Embedding dimension:", embeddings.shape[1])

    np.save(
        OUTPUT_DIR / f"finbert_{representation}_embeddings.npy",
        embeddings
    )

# Validate
assert finbert_embeddings["raw"].shape[0] == 141
assert finbert_embeddings["normalized"].shape[0] == 141
assert finbert_embeddings["ontology"].shape[0] == 141

print("\nAll FinBERT embeddings generated successfully.")
print("Device used:", device)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6741.89it/s]
[transformers] BertModel LOAD REPORT from: ProsusAI/finbert
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FinBERT loaded successfully.
Device: cuda

RAW representation
Embedding shape: (141, 768)
Embedding dimension: 768

NORMALIZED representation
Embedding shape: (141, 768)
Embedding dimension: 768

ONTOLOGY representation
Embedding shape: (141, 768)
Embedding dimension: 768

All FinBERT embeddings generated successfully.
Device used: cuda


In [20]:
# CELL 18: Compare TF-IDF, SBERT and FinBERT

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# Define concept pairs using distinctive search terms
concept_pairs = [
    ("interest income from banks", "interest income"),
    ("employee benefits expenses", "staff welfare"),
    ("sales of goods", "purchase of finished goods"),
    ("depreciation", "amortization"),
    ("interest income from banks", "interest expense")
]

# Store all embedding methods
embedding_methods = {
    "TFIDF": tfidf_embeddings,
    "SBERT": sbert_embeddings,
    "FinBERT": finbert_embeddings
}

def find_concept_index(query):

    matches = embedding_corpus[
        embedding_corpus["Normalized_Variable"].str.contains(
            query,
            case=False,
            regex=False,
            na=False
        )
    ]

    if matches.empty:
        return None

    return matches.index[0]


comparison_records = []

for concept_a, concept_b in concept_pairs:

    idx_a = find_concept_index(concept_a)
    idx_b = find_concept_index(concept_b)

    if idx_a is None or idx_b is None:
        print("Skipping unmatched pair:", concept_a, concept_b)
        continue

    if idx_a == idx_b:
        print("Skipping identical concept match:", concept_a, concept_b)
        continue

    for method, representations in embedding_methods.items():

        for representation, matrix in representations.items():

            vector_a = matrix[idx_a]
            vector_b = matrix[idx_b]

            score = cosine_similarity(
                vector_a.reshape(1, -1),
                vector_b.reshape(1, -1)
            )[0, 0]

            comparison_records.append({
                "Concept_A": embedding_corpus.loc[
                    idx_a, "Original_Variable"
                ],
                "Concept_B": embedding_corpus.loc[
                    idx_b, "Original_Variable"
                ],
                "Method": method,
                "Representation": representation,
                "Cosine_Similarity": float(score),
                "Same_Domain": (
                    embedding_corpus.loc[idx_a, "Refined_Domain"]
                    == embedding_corpus.loc[idx_b, "Refined_Domain"]
                ),
                "Same_Category": (
                    embedding_corpus.loc[idx_a, "Refined_Category"]
                    == embedding_corpus.loc[idx_b, "Refined_Category"]
                )
            })

model_comparison_df = pd.DataFrame(comparison_records)

display(
    model_comparison_df.round(4)
)

print("\nAverage similarity by method and representation:")

display(
    model_comparison_df.groupby(
        ["Method", "Representation"]
    )["Cosine_Similarity"]
    .mean()
    .unstack()
    .round(4)
)

Skipping identical concept match: interest income from banks interest income
Skipping identical concept match: depreciation amortization


,Concept_A,Concept_B,Method,Representation,Cosine_Similarity,Same_Domain,Same_Category
0,Employee benefits expenses,Staff welfare & training expenses,TFIDF,raw,0.05,True,True
1,Employee benefits expenses,Staff welfare & training expenses,TFIDF,normalized,0.05,True,True
2,Employee benefits expenses,Staff welfare & training expenses,TFIDF,ontology,0.45,True,True
3,Employee benefits expenses,Staff welfare & training expenses,SBERT,raw,0.60,True,True
4,Employee benefits expenses,Staff welfare & training expenses,SBERT,normalized,0.57,True,True
5,Employee benefits expenses,Staff welfare & training expenses,SBERT,ontology,0.86,True,True
6,Employee benefits expenses,Staff welfare & training expenses,FinBERT,raw,0.79,True,True
7,Employee benefits expenses,Staff welfare & training expenses,FinBERT,normalized,0.79,True,True
8,Employee benefits expenses,Staff welfare & training expenses,FinBERT,ontology,1.00,True,True
9,Sales of goods,Purchase of finished goods,TFIDF,raw,0.20,False,False



Average similarity by method and representation:


Representation,normalized,ontology,raw
Method,,,
FinBERT,0.84,0.95,0.84
SBERT,0.63,0.85,0.63
TFIDF,0.13,0.28,0.13


In [21]:
# CELL 19: Build a manually curated accounting concept-pair dataset

# Exact matching helper
def get_exact_concept_index(variable_name):

    matches = embedding_corpus[
        embedding_corpus["Original_Variable"].str.strip().str.lower()
        == variable_name.strip().lower()
    ]

    if len(matches) == 0:
        return None

    return matches.index[0]


# Define candidate concept relationships
candidate_pairs = [
    {
        "Concept_A": "Employee benefits expenses",
        "Concept_B": "Staff welfare & training expenses",
        "Relationship": "Equivalent/closely related",
        "Expected_Relation": 1,
        "Reason": "Both concern employee-related expenditure."
    },
    {
        "Concept_A": "Interest income from banks",
        "Concept_B": "Interest expense",
        "Relationship": "Related but different",
        "Expected_Relation": 0,
        "Reason": "Both concern interest, but have opposite accounting roles."
    },
    {
        "Concept_A": "Sales of goods",
        "Concept_B": "Purchase of finished goods",
        "Relationship": "Related but different",
        "Expected_Relation": 0,
        "Reason": "Sales and purchases represent different accounting activities."
    }
]

evaluation_records = []

for pair in candidate_pairs:

    idx_a = get_exact_concept_index(pair["Concept_A"])
    idx_b = get_exact_concept_index(pair["Concept_B"])

    if idx_a is None or idx_b is None:
        print("Concept not found:", pair)
        continue

    if idx_a == idx_b:
        print("Duplicate concept pair skipped:", pair)
        continue

    evaluation_records.append({
        **pair,
        "Index_A": idx_a,
        "Index_B": idx_b,
        "Domain_A": embedding_corpus.loc[
            idx_a, "Refined_Domain"
        ],
        "Domain_B": embedding_corpus.loc[
            idx_b, "Refined_Domain"
        ],
        "Category_A": embedding_corpus.loc[
            idx_a, "Refined_Category"
        ],
        "Category_B": embedding_corpus.loc[
            idx_b, "Refined_Category"
        ]
    })


accounting_pair_dataset = pd.DataFrame(evaluation_records)

display(accounting_pair_dataset)

print("\nValid concept pairs:", len(accounting_pair_dataset))
print("\nRelationship distribution:")
print(accounting_pair_dataset["Relationship"].value_counts())

,Concept_A,Concept_B,Relationship,Expected_Relation,Reason,Index_A,Index_B,Domain_A,Domain_B,Category_A,Category_B
0,Employee benefits expenses,Staff welfare & training expenses,Equivalent/closely related,1,Both concern employee-related expenditure.,34,28,Expenses,Expenses,Employee Costs,Employee Costs
1,Interest income from banks,Interest expense,Related but different,0,"Both concern interest, but have opposite accou...",37,4,Revenue,Expenses,Financial Revenue,Finance Costs
2,Sales of goods,Purchase of finished goods,Related but different,0,Sales and purchases represent different accoun...,26,8,Revenue,Inventory,Operating and Other Revenue,Inventory Changes



Valid concept pairs: 3

Relationship distribution:
Relationship
Related but different         2
Equivalent/closely related    1
Name: count, dtype: int64


In [22]:
# CELL 20: Generate candidate concept pairs for manual review

import pandas as pd
from itertools import combinations

candidate_records = []

for (domain, category), group in embedding_corpus.groupby(
    ["Refined_Domain", "Refined_Category"]
):

    concepts = group.reset_index(drop=True)

    # Generate pairs within the same category
    for i, j in combinations(range(len(concepts)), 2):

        candidate_records.append({
            "Concept_A": concepts.loc[i, "Original_Variable"],
            "Concept_B": concepts.loc[j, "Original_Variable"],
            "Concept_ID_A": concepts.loc[i, "Concept_ID"],
            "Concept_ID_B": concepts.loc[j, "Concept_ID"],
            "Domain_A": concepts.loc[i, "Refined_Domain"],
            "Domain_B": concepts.loc[j, "Refined_Domain"],
            "Category_A": concepts.loc[i, "Refined_Category"],
            "Category_B": concepts.loc[j, "Refined_Category"],
            "Candidate_Type": "Same category",
            "Manual_Relationship": "",
            "Manual_Label": ""
        })


# Generate pairs from different categories within the same domain
for domain, group in embedding_corpus.groupby("Refined_Domain"):

    concepts = group.reset_index(drop=True)

    for i, j in combinations(range(len(concepts)), 2):

        if concepts.loc[i, "Refined_Category"] == concepts.loc[j, "Refined_Category"]:
            continue

        candidate_records.append({
            "Concept_A": concepts.loc[i, "Original_Variable"],
            "Concept_B": concepts.loc[j, "Original_Variable"],
            "Concept_ID_A": concepts.loc[i, "Concept_ID"],
            "Concept_ID_B": concepts.loc[j, "Concept_ID"],
            "Domain_A": concepts.loc[i, "Refined_Domain"],
            "Domain_B": concepts.loc[j, "Refined_Domain"],
            "Category_A": concepts.loc[i, "Refined_Category"],
            "Category_B": concepts.loc[j, "Refined_Category"],
            "Candidate_Type": "Different category, same domain",
            "Manual_Relationship": "",
            "Manual_Label": ""
        })


candidate_pairs_df = pd.DataFrame(candidate_records)

print("Total candidate pairs:", len(candidate_pairs_df))

print("\nCandidate types:")
print(candidate_pairs_df["Candidate_Type"].value_counts())

print("\nSample candidate pairs:")
display(candidate_pairs_df.head(20))

# Save the candidate dataset
candidate_pairs_df.to_csv(
    OUTPUT_DIR / "accounting_candidate_pairs_for_review.csv",
    index=False
)

print("\nSaved: accounting_candidate_pairs_for_review.csv")

Total candidate pairs: 4321

Candidate types:
Candidate_Type
Different category, same domain    3024
Same category                      1297
Name: count, dtype: int64

Sample candidate pairs:


,Concept_A,Concept_B,Concept_ID_A,Concept_ID_B,Domain_A,Domain_B,Category_A,Category_B,Candidate_Type,Manual_Relationship,Manual_Label
0,Less: Income capitalised and transferred to DRE,Less: Interest capitalised,FIN_9C5127265BFC,FIN_A958EB1C0B22,Accounting Adjustments,Accounting Adjustments,Capitalisation and Transfers,Capitalisation and Transfers,Same category,,
1,Material/exceptional income,Gain on corporate and debt restructuring (incl...,FIN_4A28F382C7E9,FIN_96470DCFEB7E,Accounting Adjustments,Accounting Adjustments,Exceptional and Other Adjustments,Exceptional and Other Adjustments,Same category,,
2,Salaries & wages,Staff welfare & training expenses,FIN_33C5C2B371F5,FIN_3D54BB08B396,Expenses,Expenses,Employee Costs,Employee Costs,Same category,,
3,Salaries & wages,Employee benefits expenses,FIN_33C5C2B371F5,FIN_48043636822F,Expenses,Expenses,Employee Costs,Employee Costs,Same category,,
4,Salaries & wages,Gratuities and superannuation,FIN_33C5C2B371F5,FIN_61B19901962E,Expenses,Expenses,Employee Costs,Employee Costs,Same category,,
5,Salaries & wages,Payments and other expenses on employees,FIN_33C5C2B371F5,FIN_85132C1B0D2F,Expenses,Expenses,Employee Costs,Employee Costs,Same category,,
6,Salaries & wages,Share-based payments,FIN_33C5C2B371F5,FIN_9BAF6481910F,Expenses,Expenses,Employee Costs,Employee Costs,Same category,,
7,Salaries & wages,"Salaries, wages, bonus, ex gratia pf & gratuit...",FIN_33C5C2B371F5,FIN_A8CAAD939D81,Expenses,Expenses,Employee Costs,Employee Costs,Same category,,
8,Salaries & wages,Non-executive directors' fees & commission,FIN_33C5C2B371F5,FIN_C1D5258BD1D7,Expenses,Expenses,Employee Costs,Employee Costs,Same category,,
9,Salaries & wages,Less: Compensation to employees capitalised,FIN_33C5C2B371F5,FIN_DE1E55D1ED9E,Expenses,Expenses,Employee Costs,Employee Costs,Same category,,



Saved: accounting_candidate_pairs_for_review.csv


In [23]:
# CELL 21: Select a balanced candidate sample for manual review

import pandas as pd

# Reproducible sampling
RANDOM_SEED = 42

# Select same-category candidates
same_category_sample = (
    candidate_pairs_df[
        candidate_pairs_df["Candidate_Type"] == "Same category"
    ]
    .sample(n=100, random_state=RANDOM_SEED)
    .copy()
)

# Select different-category candidates within the same domain
different_category_sample = (
    candidate_pairs_df[
        candidate_pairs_df["Candidate_Type"]
        == "Different category, same domain"
    ]
    .sample(n=100, random_state=RANDOM_SEED)
    .copy()
)

# Combine and shuffle
manual_review_sample = pd.concat(
    [
        same_category_sample,
        different_category_sample
    ],
    ignore_index=True
)

manual_review_sample = manual_review_sample.sample(
    frac=1,
    random_state=RANDOM_SEED
).reset_index(drop=True)

# Assign review IDs
manual_review_sample.insert(
    0,
    "Review_ID",
    [f"PAIR_{i+1:03d}" for i in range(len(manual_review_sample))]
)

# Add annotation instructions
manual_review_sample["Manual_Relationship"] = ""
manual_review_sample["Manual_Label"] = ""
manual_review_sample["Reviewer_Notes"] = ""

# Save review file
review_path = OUTPUT_DIR / "accounting_manual_review_200_pairs.csv"

manual_review_sample.to_csv(
    review_path,
    index=False
)

print("Manual review sample created.")
print("Total pairs:", len(manual_review_sample))

print("\nCandidate distribution:")
print(manual_review_sample["Candidate_Type"].value_counts())

print("\nDomain distribution:")
print(
    manual_review_sample["Domain_A"].value_counts()
)

print("\nSample:")
display(
    manual_review_sample[
        [
            "Review_ID",
            "Concept_A",
            "Concept_B",
            "Candidate_Type",
            "Domain_A",
            "Category_A",
            "Category_B"
        ]
    ].head(20)
)

print("\nSaved to:", review_path)

Manual review sample created.
Total pairs: 200

Candidate distribution:
Candidate_Type
Same category                      100
Different category, same domain    100
Name: count, dtype: int64

Domain distribution:
Domain_A
Expenses                  159
Revenue                    37
Inventory                   2
Accounting Adjustments      1
Gains and Losses            1
Name: count, dtype: int64

Sample:


,Review_ID,Concept_A,Concept_B,Candidate_Type,Domain_A,Category_A,Category_B
0,PAIR_001,Postage & courier,Telephone expenses,Same category,Expenses,Operating and Administrative Costs,Operating and Administrative Costs
1,PAIR_002,Share-based payments,Less: Compensation to employees capitalised,Same category,Expenses,Employee Costs,Employee Costs
2,PAIR_003,Bad debts recovered,Job-work income,Same category,Revenue,Operating and Other Revenue,Operating and Other Revenue
3,PAIR_004,Rebates & discount expenses,Penalties on direct taxes,"Different category, same domain",Expenses,Operating and Administrative Costs,Tax and Regulatory Costs
4,PAIR_005,Interest expense,Selling & distribution expenses,"Different category, same domain",Expenses,Finance Costs,Operating and Administrative Costs
5,PAIR_006,Insurance premium paid,Penalties on direct taxes,"Different category, same domain",Expenses,Operating and Administrative Costs,Tax and Regulatory Costs
6,PAIR_007,Communications expenses,Rebates & discount expenses,Same category,Expenses,Operating and Administrative Costs,Operating and Administrative Costs
7,PAIR_008,Consultancy fees to auditors,Payments and other expenses on employees,"Different category, same domain",Expenses,Operating and Administrative Costs,Employee Costs
8,PAIR_009,Interest on short term borrowings / bank overd...,Excise duty,"Different category, same domain",Expenses,Finance Costs,Tax and Regulatory Costs
9,PAIR_010,Profit on securitisation/assignment of assets ...,Loss on financial instruments,Same category,Revenue,Financial Revenue,Financial Revenue



Saved to: financial_analysis_outputs\accounting_manual_review_200_pairs.csv


In [24]:
# CELL 22: Examine accounting ontology coverage

print("DOMAIN DISTRIBUTION")
print(
    embedding_corpus["Refined_Domain"]
    .value_counts()
    .to_string()
)

print("\nCATEGORY DISTRIBUTION")

category_counts = (
    embedding_corpus.groupby(
        ["Refined_Domain", "Refined_Category"]
    )
    .size()
    .reset_index(name="Concept_Count")
    .sort_values(
        ["Refined_Domain", "Concept_Count"],
        ascending=[True, False]
    )
)

display(category_counts)

print("\nCANDIDATE PAIR DISTRIBUTION BY DOMAIN")

domain_pair_counts = (
    candidate_pairs_df.groupby("Domain_A")
    .size()
    .reset_index(name="Candidate_Pair_Count")
    .sort_values("Candidate_Pair_Count", ascending=False)
)

display(domain_pair_counts)

print("\nTOTAL UNIQUE CONCEPTS:", len(embedding_corpus))
print("TOTAL CANDIDATE PAIRS:", len(candidate_pairs_df))


DOMAIN DISTRIBUTION
Refined_Domain
Expenses                  85
Revenue                   38
Gains and Losses           8
Inventory                  5
Accounting Adjustments     5

CATEGORY DISTRIBUTION


,Refined_Domain,Refined_Category,Concept_Count
0,Accounting Adjustments,Capitalisation and Transfers,2
1,Accounting Adjustments,Exceptional and Other Adjustments,2
2,Accounting Adjustments,Tax Credits and Deductions,1
7,Expenses,Operating and Administrative Costs,39
4,Expenses,Finance Costs,14
3,Expenses,Employee Costs,11
8,Expenses,Tax and Regulatory Costs,9
5,Expenses,Material and Production Costs,7
9,Expenses,Total Expenses,3
6,Expenses,Non-Cash Expenses,2



CANDIDATE PAIR DISTRIBUTION BY DOMAIN


,Domain_A,Candidate_Pair_Count
1,Expenses,3570
4,Revenue,703
2,Gains and Losses,28
0,Accounting Adjustments,10
3,Inventory,10



TOTAL UNIQUE CONCEPTS: 141
TOTAL CANDIDATE PAIRS: 4321


In [25]:
# CELL 23: Create a domain-stratified candidate review sample

import pandas as pd

RANDOM_SEED = 42
SAMPLES_PER_DOMAIN = 40

sampled_groups = []

for domain in sorted(candidate_pairs_df["Domain_A"].unique()):

    domain_data = candidate_pairs_df[
        candidate_pairs_df["Domain_A"] == domain
    ]

    # Separate the two candidate types
    same = domain_data[
        domain_data["Candidate_Type"] == "Same category"
    ]

    different = domain_data[
        domain_data["Candidate_Type"]
        == "Different category, same domain"
    ]

    # Allocate approximately half to each candidate type
    n_same = min(20, len(same))
    n_different = min(20, len(different))

    # Fill unused allocation from the other group if available
    remaining = SAMPLES_PER_DOMAIN - n_same - n_different

    if remaining > 0:
        extra_same = min(remaining, len(same) - n_same)
        n_same += extra_same
        remaining -= extra_same

    if remaining > 0:
        extra_different = min(
            remaining,
            len(different) - n_different
        )
        n_different += extra_different

    same_sample = same.sample(
        n=n_same,
        random_state=RANDOM_SEED
    ) if n_same else same.head(0)

    different_sample = different.sample(
        n=n_different,
        random_state=RANDOM_SEED
    ) if n_different else different.head(0)

    sampled_groups.append(
        pd.concat([same_sample, different_sample])
    )


balanced_review_sample = pd.concat(
    sampled_groups,
    ignore_index=True
)

balanced_review_sample = balanced_review_sample.sample(
    frac=1,
    random_state=RANDOM_SEED
).reset_index(drop=True)

# Add review identifiers and annotation columns
balanced_review_sample.insert(
    0,
    "Review_ID",
    [
        f"BAL_{i+1:03d}"
        for i in range(len(balanced_review_sample))
    ]
)

balanced_review_sample["Manual_Relationship"] = ""
balanced_review_sample["Manual_Label"] = ""
balanced_review_sample["Reviewer_Notes"] = ""

# Save
balanced_review_path = (
    OUTPUT_DIR / "balanced_accounting_review_pairs.csv"
)

balanced_review_sample.to_csv(
    balanced_review_path,
    index=False
)

print("Total selected pairs:", len(balanced_review_sample))

print("\nDistribution by domain:")
print(
    balanced_review_sample["Domain_A"].value_counts()
)

print("\nDistribution by candidate type:")
print(
    balanced_review_sample["Candidate_Type"].value_counts()
)

print("\nDomain and candidate type:")
display(
    pd.crosstab(
        balanced_review_sample["Domain_A"],
        balanced_review_sample["Candidate_Type"]
    )
)

print("\nSaved:", balanced_review_path)

Total selected pairs: 128

Distribution by domain:
Domain_A
Expenses                  40
Revenue                   40
Gains and Losses          28
Inventory                 10
Accounting Adjustments    10
Name: count, dtype: int64

Distribution by candidate type:
Candidate_Type
Same category                      65
Different category, same domain    63
Name: count, dtype: int64

Domain and candidate type:


Candidate_Type,"Different category, same domain",Same category
Domain_A,,
Accounting Adjustments,8,2
Expenses,20,20
Gains and Losses,15,13
Inventory,0,10
Revenue,20,20



Saved: financial_analysis_outputs\balanced_accounting_review_pairs.csv


In [26]:
# CELL 24: Prepare final manual annotation template

annotation_df = balanced_review_sample[
    [
        "Review_ID",
        "Concept_ID_A",
        "Concept_A",
        "Domain_A",
        "Category_A",
        "Concept_ID_B",
        "Concept_B",
        "Domain_B",
        "Category_B",
        "Candidate_Type"
    ]
].copy()

# Reviewer annotation fields
annotation_df["Relationship_Label"] = ""
annotation_df["Binary_Label"] = ""
annotation_df["Reviewer_Confidence"] = ""
annotation_df["Reviewer_Notes"] = ""

# Save the annotation template
annotation_path = (
    OUTPUT_DIR / "accounting_relationship_annotation_template.csv"
)

annotation_df.to_csv(
    annotation_path,
    index=False
)

print("Annotation template created.")
print("Number of pairs:", len(annotation_df))

print("\nColumns:")
print(annotation_df.columns.tolist())

print("\nSample:")
display(annotation_df.head(10))

print("\nSaved:", annotation_path)


Annotation template created.
Number of pairs: 128

Columns:
['Review_ID', 'Concept_ID_A', 'Concept_A', 'Domain_A', 'Category_A', 'Concept_ID_B', 'Concept_B', 'Domain_B', 'Category_B', 'Candidate_Type', 'Relationship_Label', 'Binary_Label', 'Reviewer_Confidence', 'Reviewer_Notes']

Sample:


,Review_ID,Concept_ID_A,Concept_A,Domain_A,Category_A,Concept_ID_B,Concept_B,Domain_B,Category_B,Candidate_Type,Relationship_Label,Binary_Label,Reviewer_Confidence,Reviewer_Notes
0,BAL_001,FIN_0853F25AD750,Gain on disposal of non-current non-financial ...,Gains and Losses,Asset and Investment Gains,FIN_F9DF1D350BC6,"Profit on sale of investment in subsidiary, as...",Gains and Losses,Asset and Investment Gains,Same category,,,,
1,BAL_002,FIN_33A8A4C6DA7B,Environment and pollution control related expe...,Expenses,Operating and Administrative Costs,FIN_3ECC6CDD70CC,Total expenses of continued operations,Expenses,Total Expenses,"Different category, same domain",,,,
2,BAL_003,FIN_12AD529248EC,Social and community expenses (including CSR exp),Expenses,Operating and Administrative Costs,FIN_47F8AD6236DC,Consultancy fees to others,Expenses,Operating and Administrative Costs,Same category,,,,
3,BAL_004,FIN_03072DDDEA19,Interest on short term borrowings / bank overd...,Expenses,Finance Costs,FIN_FF7C3633B08F,Donations,Expenses,Operating and Administrative Costs,"Different category, same domain",,,,
4,BAL_005,FIN_3819CC19E4FC,Trading revenue and non-financial services,Revenue,Operating and Other Revenue,FIN_B2A392E17F73,Other income,Revenue,Operating and Other Revenue,Same category,,,,
5,BAL_006,FIN_0853F25AD750,Gain on disposal of non-current non-financial ...,Gains and Losses,Asset and Investment Gains,FIN_D547EC0DEFF3,Insurance claims,Gains and Losses,Asset and Investment Gains,Same category,,,,
6,BAL_007,FIN_0853F25AD750,Gain on disposal of non-current non-financial ...,Gains and Losses,Asset and Investment Gains,FIN_FA0FC6B6D418,Of which: Share of loss in partnership and JV ...,Gains and Losses,Financial Gains and Losses,"Different category, same domain",,,,
7,BAL_008,FIN_BEB60B8338C0,Income from non-financial services,Revenue,Operating and Other Revenue,FIN_D06A42F73B52,Of which: Export incentives including duty dra...,Revenue,Operating and Other Revenue,Same category,,,,
8,BAL_009,FIN_18A52D241144,Purchase of finished goods,Inventory,Inventory Changes,FIN_2DA061F58EF8,Change in stock of wip and semifinished goods ...,Inventory,Inventory Changes,Same category,,,,
9,BAL_010,FIN_DDF2D1E61E18,Auditors fees for company law matters & others,Expenses,Operating and Administrative Costs,FIN_F6C272E92454,Distribution expenses (including outward freight),Expenses,Operating and Administrative Costs,Same category,,,,



Saved: financial_analysis_outputs\accounting_relationship_annotation_template.csv


In [27]:

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

sbert_matrix = cosine_similarity(sbert_embeddings["raw"])
finbert_matrix = cosine_similarity(finbert_embeddings["raw"])

domain_values = embedding_corpus["Refined_Domain"].values
category_values = embedding_corpus["Refined_Category"].values
qualifier_values = embedding_corpus["Refined_Qualifier"].values

domain_matrix = (
    domain_values[:, None] == domain_values[None, :]
).astype(float)

category_matrix = (
    category_values[:, None] == category_values[None, :]
).astype(float)

qualifier_matrix = (
    qualifier_values[:, None] == qualifier_values[None, :]
).astype(float)

semantic_matrix = (
    0.5 * sbert_matrix +
    0.5 * finbert_matrix
)

hybrid_similarity = (
    0.50 * semantic_matrix +
    0.20 * domain_matrix +
    0.20 * category_matrix +
    0.10 * qualifier_matrix
)

np.fill_diagonal(hybrid_similarity, 1.0)

np.save(
    OUTPUT_DIR / "ontology_aware_hybrid_similarity.npy",
    hybrid_similarity
)

print("Hybrid similarity matrix:", hybrid_similarity.shape)
print("Minimum similarity:", hybrid_similarity.min())
print("Maximum similarity:", hybrid_similarity.max())
print("Mean similarity:", hybrid_similarity.mean())

print("Saved ontology-aware hybrid similarity matrix.")


Hybrid similarity matrix: (141, 141)
Minimum similarity: 0.11883943527936935
Maximum similarity: 1.0
Mean similarity: 0.43191538738998136
Saved ontology-aware hybrid similarity matrix.


In [28]:

def retrieve_hybrid_neighbours(query, top_n=10):

    matches = embedding_corpus[
        embedding_corpus["Normalized_Variable"].str.contains(
            query,
            case=False,
            regex=False,
            na=False
        )
    ]

    if matches.empty:
        print("Concept not found:", query)
        return

    query_idx = matches.index[0]
    scores = hybrid_similarity[query_idx].copy()
    scores[query_idx] = -1

    top_indices = np.argsort(scores)[::-1][:top_n]

    results = embedding_corpus.iloc[top_indices][
        [
            "Concept_ID",
            "Original_Variable",
            "Refined_Domain",
            "Refined_Category",
            "Refined_Qualifier"
        ]
    ].copy()

    results["Hybrid_Similarity"] = scores[top_indices]

    print("Query:", matches.iloc[0]["Original_Variable"])
    display(results.reset_index(drop=True))


retrieve_hybrid_neighbours("interest income")
retrieve_hybrid_neighbours("employee benefits")
retrieve_hybrid_neighbours("sales of goods")
retrieve_hybrid_neighbours("depreciation")


Query: Interest income from banks


,Concept_ID,Original_Variable,Refined_Domain,Refined_Category,Refined_Qualifier,Hybrid_Similarity
0,FIN_E3EF503E4328,Interest income,Revenue,Financial Revenue,Primary,0.92
1,FIN_FA909C160A84,Interest income of cos other than banks from i...,Revenue,Financial Revenue,Primary,0.89
2,FIN_8A0045F3B426,Interest income of cos other than banks from l...,Revenue,Financial Revenue,Primary,0.88
3,FIN_8523C43B68BA,Income from financial services,Revenue,Financial Revenue,Primary,0.87
4,FIN_A7DB8D3B7E24,Fee based financial services income,Revenue,Financial Revenue,Primary,0.84
5,FIN_BC20C36A5E0C,Other financial services income,Revenue,Financial Revenue,Primary,0.83
6,FIN_FAF037CFF47C,Income from treasury operations,Revenue,Financial Revenue,Primary,0.82
7,FIN_AEF2B61C625A,Profit on securitisation/assignment of assets ...,Revenue,Financial Revenue,Primary,0.82
8,FIN_77693C509B90,Dividend income,Revenue,Financial Revenue,Primary,0.81
9,FIN_328370A41467,Gain on financial instruments,Revenue,Financial Revenue,Primary,0.77


Query: Employee benefits expenses


,Concept_ID,Original_Variable,Refined_Domain,Refined_Category,Refined_Qualifier,Hybrid_Similarity
0,FIN_85132C1B0D2F,Payments and other expenses on employees,Expenses,Employee Costs,Primary,0.89
1,FIN_3D54BB08B396,Staff welfare & training expenses,Expenses,Employee Costs,Primary,0.85
2,FIN_A8CAAD939D81,"Salaries, wages, bonus, ex gratia pf & gratuit...",Expenses,Employee Costs,Primary,0.78
3,FIN_33C5C2B371F5,Salaries & wages,Expenses,Employee Costs,Primary,0.77
4,FIN_C1D5258BD1D7,Non-executive directors' fees & commission,Expenses,Employee Costs,Primary,0.77
5,FIN_E2E6A58D2276,Contribution to provident fund,Expenses,Employee Costs,Primary,0.74
6,FIN_61B19901962E,Gratuities and superannuation,Expenses,Employee Costs,Primary,0.74
7,FIN_9BAF6481910F,Share-based payments,Expenses,Employee Costs,Primary,0.71
8,FIN_DE1E55D1ED9E,Less: Compensation to employees capitalised,Expenses,Employee Costs,Deduction,0.69
9,FIN_F69DC252933E,Bonus & ex-gratia,Expenses,Employee Costs,Primary,0.69


Query: Sales of goods


,Concept_ID,Original_Variable,Refined_Domain,Refined_Category,Refined_Qualifier,Hybrid_Similarity
0,FIN_958A75FAB61F,Manufacturing revenue,Revenue,Operating and Other Revenue,Primary,0.86
1,FIN_E59E95731018,"Sale of electricity, gas and water",Revenue,Operating and Other Revenue,Primary,0.82
2,FIN_3819CC19E4FC,Trading revenue and non-financial services,Revenue,Operating and Other Revenue,Primary,0.81
3,FIN_D283F080F0A7,Income from repairs & maintenance including af...,Revenue,Operating and Other Revenue,Primary,0.77
4,FIN_B2A392E17F73,Other income,Revenue,Operating and Other Revenue,Primary,0.77
5,FIN_DDFB8304B675,Repairs & maintenance of vehicles & others,Revenue,Operating and Other Revenue,Primary,0.75
6,FIN_1CC010643954,Fiscal benefits,Revenue,Operating and Other Revenue,Primary,0.75
7,FIN_4BC68738CC96,Repairs & maintenance,Revenue,Operating and Other Revenue,Primary,0.74
8,FIN_4730462E2E75,Repairs & maintenance of plant & machinery,Revenue,Operating and Other Revenue,Primary,0.74
9,FIN_BEB60B8338C0,Income from non-financial services,Revenue,Operating and Other Revenue,Primary,0.74


Query: Expenses other than Depreciation, Interest, Taxes, Provisions and Amortizations


,Concept_ID,Original_Variable,Refined_Domain,Refined_Category,Refined_Qualifier,Hybrid_Similarity
0,FIN_D7A4B00FD879,Provisions/impairment and credit balances writ...,Expenses,Non-Cash Expenses,Primary,0.79
1,FIN_78A126F37B09,Other interest expense (term not specified),Expenses,Finance Costs,Primary,0.67
2,FIN_8AF75BA118BF,Other operational expenses of non-financial se...,Expenses,Operating and Administrative Costs,Primary,0.66
3,FIN_9FFC7914BF58,Total expenses excluding direct taxes,Expenses,Total Expenses,Primary,0.66
4,FIN_FF18FCEBF80B,Financial services expenses,Expenses,Finance Costs,Primary,0.64
5,FIN_0BD8ED8EC598,Interest expense,Expenses,Finance Costs,Primary,0.64
6,FIN_5564F8EC361B,Fee based financial services expenses,Expenses,Finance Costs,Primary,0.64
7,FIN_903FDBA79BF6,Selling & distribution expenses,Expenses,Operating and Administrative Costs,Primary,0.64
8,FIN_7CBFA4B52D50,Miscellaneous expenditure,Expenses,Operating and Administrative Costs,Primary,0.64
9,FIN_22F2FB4E6109,Other fund based financial services expenses,Expenses,Finance Costs,Primary,0.64


In [31]:

import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.preprocessing import normalize

sbert_matrix = normalize(
    sbert_embeddings["raw"]
)

finbert_matrix = normalize(
    finbert_embeddings["raw"]
)

hybrid_matrix = normalize(
    np.concatenate(
        [sbert_matrix, finbert_matrix],
        axis=1
    )
)

clustering_inputs = {
    "SBERT": sbert_matrix,
    "FinBERT": finbert_matrix,
    "Hybrid": hybrid_matrix
}

clustering_results = []
cluster_assignments = {}

N_CLUSTERS = 8

for method, matrix in clustering_inputs.items():

    model = KMeans(
        n_clusters=N_CLUSTERS,
        random_state=42,
        n_init=20
    )

    labels = model.fit_predict(matrix)

    silhouette = silhouette_score(
        matrix,
        labels
    )

    db_index = davies_bouldin_score(
        matrix,
        labels
    )

    cluster_assignments[method] = labels

    clustering_results.append({
        "Method": method,
        "Clusters": N_CLUSTERS,
        "Silhouette_Score": silhouette,
        "Davies_Bouldin_Index": db_index,
        "Inertia": model.inertia_
    })

clustering_comparison = pd.DataFrame(
    clustering_results
)

print("EMBEDDING DIMENSIONS")
print("SBERT:", sbert_matrix.shape)
print("FinBERT:", finbert_matrix.shape)
print("Hybrid:", hybrid_matrix.shape)

print("\nCLUSTERING PERFORMANCE")
display(clustering_comparison.round(4))

clustered_concepts = embedding_corpus.copy()

clustered_concepts["SBERT_Cluster"] = cluster_assignments["SBERT"]
clustered_concepts["FinBERT_Cluster"] = cluster_assignments["FinBERT"]
clustered_concepts["Hybrid_Cluster"] = cluster_assignments["Hybrid"]

clustered_concepts.to_csv(
    OUTPUT_DIR / "financial_concept_clusters.csv",
    index=False
)

np.save(
    OUTPUT_DIR / "hybrid_embeddings.npy",
    hybrid_matrix
)

print("\nTotal concepts clustered:", len(clustered_concepts))
print("Hybrid embedding dimensions:", hybrid_matrix.shape[1])
print("Results saved successfully.")

EMBEDDING DIMENSIONS
SBERT: (141, 384)
FinBERT: (141, 768)
Hybrid: (141, 1152)

CLUSTERING PERFORMANCE


,Method,Clusters,Silhouette_Score,Davies_Bouldin_Index,Inertia
0,SBERT,8,0.08,2.58,79.43
1,FinBERT,8,0.05,2.70,30.36
2,Hybrid,8,0.06,2.81,57.48



Total concepts clustered: 141
Hybrid embedding dimensions: 1152
Results saved successfully.


In [32]:

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

cluster_count_results = []

for method, matrix in clustering_inputs.items():

    for k in range(2, 13):

        model = KMeans(
            n_clusters=k,
            random_state=42,
            n_init=20
        )

        labels = model.fit_predict(matrix)

        silhouette = silhouette_score(matrix, labels)
        db_index = davies_bouldin_score(matrix, labels)

        cluster_count_results.append({
            "Method": method,
            "Number_of_Clusters": k,
            "Silhouette_Score": silhouette,
            "Davies_Bouldin_Index": db_index,
            "Inertia": model.inertia_
        })

cluster_count_results = pd.DataFrame(cluster_count_results)

display(
    cluster_count_results.sort_values(
        ["Method", "Silhouette_Score"],
        ascending=[True, False]
    ).round(4)
)

best_cluster_counts = (
    cluster_count_results
    .sort_values("Silhouette_Score", ascending=False)
    .groupby("Method")
    .head(1)
    .reset_index(drop=True)
)

print("BEST CLUSTER COUNT FOR EACH METHOD")
display(best_cluster_counts.round(4))

cluster_count_results.to_csv(
    OUTPUT_DIR / "cluster_count_comparison.csv",
    index=False
)


,Method,Number_of_Clusters,Silhouette_Score,Davies_Bouldin_Index,Inertia
12,FinBERT,3,0.08,2.80,35.34
11,FinBERT,2,0.08,3.30,37.87
20,FinBERT,11,0.06,2.42,28.25
15,FinBERT,6,0.06,2.82,31.83
19,FinBERT,10,0.06,2.56,28.97
17,FinBERT,8,0.05,2.70,30.36
13,FinBERT,4,0.05,3.02,33.90
18,FinBERT,9,0.05,2.62,29.61
21,FinBERT,12,0.05,2.54,27.71
16,FinBERT,7,0.05,2.68,30.98


BEST CLUSTER COUNT FOR EACH METHOD


,Method,Number_of_Clusters,Silhouette_Score,Davies_Bouldin_Index,Inertia
0,SBERT,10,0.08,2.46,76.24
1,FinBERT,3,0.08,2.80,35.34
2,Hybrid,10,0.08,2.56,54.24


In [33]:

from sklearn.cluster import KMeans

best_cluster_models = {}
best_cluster_assignments = {}

for _, row in best_cluster_counts.iterrows():

    method = row["Method"]
    k = int(row["Number_of_Clusters"])

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    labels = model.fit_predict(clustering_inputs[method])

    best_cluster_models[method] = model
    best_cluster_assignments[method] = labels

cluster_inspection = embedding_corpus.copy()

for method, labels in best_cluster_assignments.items():
    cluster_inspection[f"{method}_Best_Cluster"] = labels

for method in best_cluster_assignments:

    print("\n" + "=" * 80)
    print("CLUSTER INSPECTION:", method)
    print("=" * 80)

    column = f"{method}_Best_Cluster"

    for cluster_id in sorted(cluster_inspection[column].unique()):

        subset = cluster_inspection[
            cluster_inspection[column] == cluster_id
        ]

        print(
            f"\nCluster {cluster_id} | "
            f"Concepts: {len(subset)}"
        )

        display(
            subset[
                [
                    "Original_Variable",
                    "Refined_Domain",
                    "Refined_Category"
                ]
            ].sort_values(
                ["Refined_Domain", "Refined_Category"]
            ).reset_index(drop=True)
        )

cluster_inspection.to_csv(
    OUTPUT_DIR / "best_financial_concept_cluster_inspection.csv",
    index=False
)

print("\nCluster inspection saved successfully.")



CLUSTER INSPECTION: SBERT

Cluster 0 | Concepts: 6


,Original_Variable,Refined_Domain,Refined_Category
0,Power & fuel (including wheeling charges paid ...,Expenses,Material and Production Costs
1,"Power, fuel (including wheeling charges paid b...",Expenses,Material and Production Costs
2,Water charges,Expenses,Operating and Administrative Costs
3,Environment and pollution control related expe...,Expenses,Operating and Administrative Costs
4,Rent & lease rent,Expenses,Operating and Administrative Costs
5,"Sale of electricity, gas and water",Revenue,Operating and Other Revenue



Cluster 1 | Concepts: 34


,Original_Variable,Refined_Domain,Refined_Category
0,Staff welfare & training expenses,Expenses,Employee Costs
1,Employee benefits expenses,Expenses,Employee Costs
2,Payments and other expenses on employees,Expenses,Employee Costs
3,Other fund based financial services expenses,Expenses,Finance Costs
4,Fee based financial services expenses,Expenses,Finance Costs
5,Bill discounting charges,Expenses,Finance Costs
6,Fund based financial services expenses,Expenses,Finance Costs
7,Financial services expenses,Expenses,Finance Costs
8,Raw material expenses,Expenses,Material and Production Costs
9,"Expenses other than Depreciation, Interest, Ta...",Expenses,Non-Cash Expenses



Cluster 2 | Concepts: 23


,Original_Variable,Refined_Domain,Refined_Category
0,Less: Income capitalised and transferred to DRE,Accounting Adjustments,Capitalisation and Transfers
1,Material/exceptional income,Accounting Adjustments,Exceptional and Other Adjustments
2,Salaries & wages,Expenses,Employee Costs
3,Gratuities and superannuation,Expenses,Employee Costs
4,"Salaries, wages, bonus, ex gratia pf & gratuit...",Expenses,Employee Costs
5,Contribution to provident fund,Expenses,Employee Costs
6,Bonus & ex-gratia,Expenses,Employee Costs
7,Outsourced industrial jobs (Including Mfg.),Expenses,Material and Production Costs
8,Royalty,Expenses,Operating and Administrative Costs
9,Outsourced professional services,Expenses,Operating and Administrative Costs


... (37 outputs left out)

,Original_Variable,Refined_Domain,Refined_Category
0,Less: Interest capitalised,Accounting Adjustments,Capitalisation and Transfers
1,Interest on short term borrowings / bank overd...,Expenses,Finance Costs
2,Interest expense,Expenses,Finance Costs
3,Other interest expense (term not specified),Expenses,Finance Costs
4,Other borrowing costs,Expenses,Finance Costs
5,Interest on finance lease,Expenses,Finance Costs
6,Interest on long term borrowings / convertible...,Expenses,Finance Costs
7,Rent & lease rent,Expenses,Operating and Administrative Costs
8,Interest income from banks,Revenue,Financial Revenue
9,Interest income of cos other than banks from l...,Revenue,Financial Revenue



Cluster 9 | Concepts: 3


,Original_Variable,Refined_Domain,Refined_Category
0,Power & fuel (including wheeling charges paid ...,Expenses,Material and Production Costs
1,"Power, fuel (including wheeling charges paid b...",Expenses,Material and Production Costs
2,"Sale of electricity, gas and water",Revenue,Operating and Other Revenue



Cluster inspection saved successfully.


In [34]:

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

evaluation_pairs = [
    {
        "Concept_A": "Employee benefits expenses",
        "Concept_B": "Staff welfare & training expenses",
        "Expected_Relationship": "Equivalent or closely related",
        "Expected_Label": 1
    },
    {
        "Concept_A": "Interest income from banks",
        "Concept_B": "Interest expense",
        "Expected_Relationship": "Related but different",
        "Expected_Label": 0
    },
    {
        "Concept_A": "Sales of goods",
        "Concept_B": "Purchase of finished goods",
        "Expected_Relationship": "Related but different",
        "Expected_Label": 0
    }
]

def find_concept_index(name):
    matches = financial_dictionary[
        financial_dictionary["Original_Variable"].str.lower()
        == name.lower()
    ]

    if len(matches) == 0:
        raise ValueError(f"Concept not found: {name}")

    concept_id = matches.iloc[0]["Concept_ID"]

    return embedding_corpus.index[
        embedding_corpus["Concept_ID"] == concept_id
    ][0]

evaluation_results = []

for pair in evaluation_pairs:

    idx_a = find_concept_index(pair["Concept_A"])
    idx_b = find_concept_index(pair["Concept_B"])

    result = {
        "Concept_A": pair["Concept_A"],
        "Concept_B": pair["Concept_B"],
        "Expected_Relationship": pair["Expected_Relationship"],
        "Expected_Label": pair["Expected_Label"]
    }

    for method, matrix in clustering_inputs.items():

        similarity = cosine_similarity(
            matrix[idx_a:idx_a+1],
            matrix[idx_b:idx_b+1]
        )[0, 0]

        result[f"{method}_Similarity"] = similarity

    evaluation_results.append(result)

role_aware_diagnostic = pd.DataFrame(evaluation_results)

display(role_aware_diagnostic.round(4))

role_aware_diagnostic.to_csv(
    OUTPUT_DIR / "accounting_role_aware_diagnostic.csv",
    index=False
)

print("Preliminary accounting-role diagnostic completed.")


,Concept_A,Concept_B,Expected_Relationship,Expected_Label,SBERT_Similarity,FinBERT_Similarity,Hybrid_Similarity
0,Employee benefits expenses,Staff welfare & training expenses,Equivalent or closely related,1,0.60,0.79,0.69
1,Interest income from banks,Interest expense,Related but different,0,0.57,0.88,0.72
2,Sales of goods,Purchase of finished goods,Related but different,0,0.73,0.86,0.80


Preliminary accounting-role diagnostic completed.


In [35]:

import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import cosine_similarity

concepts = embedding_corpus.copy().reset_index(drop=True)

sbert_sim = cosine_similarity(
    sbert_embeddings["raw"]
)

finbert_sim = cosine_similarity(
    finbert_embeddings["raw"]
)

semantic_sim = (
    0.5 * sbert_sim +
    0.5 * finbert_sim
)

domain_sim = np.equal.outer(
    concepts["Refined_Domain"].values,
    concepts["Refined_Domain"].values
).astype(float)

category_sim = np.equal.outer(
    concepts["Refined_Category"].values,
    concepts["Refined_Category"].values
).astype(float)

qualifier_values = concepts["Refined_Qualifier"].fillna(
    "Unknown"
).values

qualifier_sim = np.equal.outer(
    qualifier_values,
    qualifier_values
).astype(float)

def accounting_role(text):
    text = str(text).lower()

    if "income" in text or "revenue" in text or "sales" in text:
        return "Income"

    if "expense" in text or "cost" in text or "charges" in text:
        return "Expense"

    if "gain" in text or "profit" in text:
        return "Gain"

    if "loss" in text:
        return "Loss"

    if "less:" in text or "deduction" in text or "credits" in text:
        return "Adjustment"

    if "change in stock" in text or "inventory" in text:
        return "Inventory"

    return "Other"

concepts["Accounting_Role"] = concepts[
    "Original_Variable"
].apply(accounting_role)

role_values = concepts["Accounting_Role"].values

role_sim = np.equal.outer(
    role_values,
    role_values
).astype(float)

role_aware_similarity = (
    0.35 * semantic_sim +
    0.15 * domain_sim +
    0.15 * category_sim +
    0.10 * qualifier_sim +
    0.25 * role_sim
)

np.fill_diagonal(role_aware_similarity, 1.0)

print("Role-aware similarity matrix:", role_aware_similarity.shape)

print("\nAccounting role distribution:")
display(
    concepts["Accounting_Role"].value_counts().to_frame("Concept_Count")
)

np.save(
    OUTPUT_DIR / "accounting_role_aware_similarity.npy",
    role_aware_similarity
)

concepts.to_csv(
    OUTPUT_DIR / "financial_concepts_with_accounting_roles.csv",
    index=False
)

print("\nAccounting-role-aware similarity model constructed.")


Role-aware similarity matrix: (141, 141)

Accounting role distribution:


,Concept_Count
Accounting_Role,
Other,55
Expense,41
Income,26
Gain,9
Loss,4
Inventory,3
Adjustment,3



Accounting-role-aware similarity model constructed.


In [36]:

import re
import numpy as np
import pandas as pd

def classify_accounting_role(row):

    text = str(row["Original_Variable"]).lower()
    domain = str(row["Refined_Domain"]).lower()
    category = str(row["Refined_Category"]).lower()

    if domain == "inventory":
        return "Inventory"

    if domain == "accounting adjustments":
        return "Adjustment"

    if domain == "gains and losses":
        if re.search(r"\bloss\b|losses|impairment", text):
            return "Loss"
        if re.search(r"\bgain\b|gains|profit", text):
            return "Gain"
        return "Other"

    if domain == "expenses":
        return "Expense"

    if domain == "revenue":
        if re.search(r"\bloss\b|losses", text):
            return "Loss"
        if re.search(r"\bgain\b|gains|profit on sale", text):
            return "Gain"
        return "Income"

    if re.search(r"\binventory\b|change in stock|finished goods", text):
        return "Inventory"

    if re.search(r"\bloss\b|losses|impairment", text):
        return "Loss"

    if re.search(r"\bgain\b|gains", text):
        return "Gain"

    if re.search(r"\bincome\b|revenue|sales|dividend", text):
        return "Income"

    if re.search(r"expense|expenditure|charges|cost|fees|salaries|wages|paid|consumed|borrowings", text):
        return "Expense"

    if re.search(r"less:|capitalised|capitalized|tax credits|deductions", text):
        return "Adjustment"

    return "Other"


concepts = embedding_corpus.copy().reset_index(drop=True)

concepts["Accounting_Role"] = concepts.apply(
    classify_accounting_role,
    axis=1
)

role_distribution = (
    concepts["Accounting_Role"]
    .value_counts()
    .rename_axis("Accounting_Role")
    .reset_index(name="Concept_Count")
)

display(role_distribution)

print(
    "Concepts classified:",
    concepts["Accounting_Role"].ne("Other").sum(),
    "out of",
    len(concepts)
)

print(
    "Unclassified concepts:",
    concepts.loc[
        concepts["Accounting_Role"] == "Other",
        "Original_Variable"
    ].tolist()
)

role_values = concepts["Accounting_Role"].values

role_sim = np.equal.outer(
    role_values,
    role_values
).astype(float)

sbert_sim = cosine_similarity(sbert_embeddings["raw"])
finbert_sim = cosine_similarity(finbert_embeddings["raw"])

semantic_sim = 0.5 * sbert_sim + 0.5 * finbert_sim

domain_sim = np.equal.outer(
    concepts["Refined_Domain"].values,
    concepts["Refined_Domain"].values
).astype(float)

category_sim = np.equal.outer(
    concepts["Refined_Category"].values,
    concepts["Refined_Category"].values
).astype(float)

qualifier_values = concepts["Refined_Qualifier"].fillna("Unknown").values

qualifier_sim = np.equal.outer(
    qualifier_values,
    qualifier_values
).astype(float)

role_aware_similarity = (
    0.35 * semantic_sim +
    0.15 * domain_sim +
    0.15 * category_sim +
    0.10 * qualifier_sim +
    0.25 * role_sim
)

np.fill_diagonal(role_aware_similarity, 1.0)

np.save(
    OUTPUT_DIR / "accounting_role_aware_similarity.npy",
    role_aware_similarity
)

concepts.to_csv(
    OUTPUT_DIR / "financial_concepts_with_accounting_roles.csv",
    index=False
)

print("\nUpdated role-aware similarity matrix:", role_aware_similarity.shape)
print("Updated accounting-role classification completed.")


,Accounting_Role,Concept_Count
0,Expense,85
1,Income,35
2,Gain,6
3,Inventory,5
4,Adjustment,5
5,Loss,4
6,Other,1


Concepts classified: 140 out of 141
Unclassified concepts: ['Insurance claims']

Updated role-aware similarity matrix: (141, 141)
Updated accounting-role classification completed.


In [37]:

import numpy as np
import pandas as pd

def find_concept(query):
    matches = concepts[
        concepts["Original_Variable"].str.contains(
            query,
            case=False,
            regex=False,
            na=False
        )
    ]

    if len(matches) == 0:
        return None, None

    return matches.index[0], matches.iloc[0]["Original_Variable"]


diagnostic_pairs = [
    ("Employee benefits", "Staff welfare", 1, "Closely related"),
    ("Interest income from banks", "Interest expense", 0, "Different accounting roles"),
    ("Sales of goods", "Purchase of finished goods", 0, "Different accounting roles")
]

results = []

for query_a, query_b, expected, relationship in diagnostic_pairs:

    idx_a, name_a = find_concept(query_a)
    idx_b, name_b = find_concept(query_b)

    if idx_a is None or idx_b is None:
        print("Pair not found:", query_a, query_b)
        continue

    results.append({
        "Concept_A": name_a,
        "Concept_B": name_b,
        "Expected_Relationship": relationship,
        "Expected_Label": expected,
        "Original_Hybrid": round(float(hybrid_similarity[idx_a, idx_b]), 4),
        "Role_Aware": round(float(role_aware_similarity[idx_a, idx_b]), 4),
        "Role_A": concepts.loc[idx_a, "Accounting_Role"],
        "Role_B": concepts.loc[idx_b, "Accounting_Role"]
    })

comparison_df = pd.DataFrame(results)

display(comparison_df)

comparison_df.to_csv(
    OUTPUT_DIR / "diagnostic_model_comparison.csv",
    index=False
)

print("Diagnostic pairs evaluated:", len(comparison_df))
print("Results saved.")


,Concept_A,Concept_B,Expected_Relationship,Expected_Label,Original_Hybrid,Role_Aware,Role_A,Role_B
0,Employee benefits expenses,Staff welfare & training expenses,Closely related,1,0.85,0.89,Expense,Expense
1,Interest income from banks,Interest expense,Different accounting roles,0,0.46,0.35,Income,Expense
2,Sales of goods,Purchase of finished goods,Different accounting roles,0,0.50,0.38,Income,Inventory


Diagnostic pairs evaluated: 3
Results saved.


In [38]:

import pandas as pd
import numpy as np

data_path = OUTPUT_DIR / "unified_consolidated_financial_data.csv"

financial_data = pd.read_csv(data_path, low_memory=False)

identifier_columns = ["Company_Key", "Company Name"]
feature_columns = [
    col for col in financial_data.columns
    if col not in identifier_columns
]

numeric_data = financial_data[feature_columns].copy()

for col in feature_columns:
    numeric_data[col] = pd.to_numeric(
        numeric_data[col].astype(str).str.replace(",", "", regex=False).str.strip(),
        errors="coerce"
    )

feature_coverage = pd.DataFrame({
    "Variable": feature_columns,
    "Available_Values": numeric_data.notna().sum().values,
    "Missing_Values": numeric_data.isna().sum().values,
    "Coverage_Percent": (numeric_data.notna().mean().values * 100).round(2),
    "Missing_Percent": (numeric_data.isna().mean().values * 100).round(2)
})

feature_coverage = feature_coverage.sort_values(
    "Coverage_Percent",
    ascending=False
).reset_index(drop=True)

company_coverage = pd.DataFrame({
    "Company_Key": financial_data["Company_Key"],
    "Company_Name": financial_data["Company Name"],
    "Available_Features": numeric_data.notna().sum(axis=1).values,
    "Missing_Features": numeric_data.isna().sum(axis=1).values,
    "Coverage_Percent": (numeric_data.notna().mean(axis=1).values * 100).round(2)
})

print("DATASET OVERVIEW")
print("Companies:", len(financial_data))
print("Financial variables:", len(feature_columns))
print("Total possible values:", numeric_data.size)
print("Available values:", int(numeric_data.notna().sum().sum()))
print("Overall coverage:", round(numeric_data.notna().mean().mean() * 100, 2), "%")
print("Overall missingness:", round(numeric_data.isna().mean().mean() * 100, 2), "%")

print("\nTOP 20 MOST REPORTED VARIABLES")
display(feature_coverage.head(20))

print("\nTOP 20 LEAST REPORTED VARIABLES")
display(feature_coverage.tail(20))

print("\nCOMPANY COVERAGE DISTRIBUTION")
display(
    company_coverage["Available_Features"]
    .describe()
    .to_frame("Available_Features")
)

print("\nCOMPANIES WITH THE MOST AVAILABLE FINANCIAL VARIABLES")
display(
    company_coverage.sort_values(
        "Available_Features",
        ascending=False
    ).head(15)
)

feature_coverage.to_csv(
    OUTPUT_DIR / "financial_feature_coverage.csv",
    index=False
)

company_coverage.to_csv(
    OUTPUT_DIR / "financial_company_coverage.csv",
    index=False
)

numeric_data.to_csv(
    OUTPUT_DIR / "financial_numeric_matrix.csv",
    index=False
)

print("\nDataset quality analysis saved.")


DATASET OVERVIEW
Companies: 13162
Financial variables: 182
Total possible values: 2395484
Available values: 135902
Overall coverage: 5.67 %
Overall missingness: 94.33 %

TOP 20 MOST REPORTED VARIABLES


,Variable,Available_Values,Missing_Values,Coverage_Percent,Missing_Percent
0,PL__Total Income from continued operations,1772,11390,13.46,86.54
1,CHANGE__Total expenses excluding direct taxes,1772,11390,13.46,86.54
2,CHANGE__Employee benefits expenses,1771,11391,13.46,86.54
3,CHANGE__Ordinary and non-exceptional expenses,1772,11390,13.46,86.54
4,"CHANGE__Expenses other than Depreciation, Inte...",1772,11390,13.46,86.54
5,"CHANGE__Salaries, wages, bonus, ex gratia pf &...",1771,11391,13.46,86.54
6,CHANGE__Total Income from continued operations,1772,11390,13.46,86.54
7,CHANGE__Total expenses of continued operations,1772,11390,13.46,86.54
8,CHANGE__Salaries & wages,1770,11392,13.45,86.55
9,CHANGE__Miscellaneous expenditure,1765,11397,13.41,86.59



TOP 20 LEAST REPORTED VARIABLES


,Variable,Available_Values,Missing_Values,Coverage_Percent,Missing_Percent
162,PL__Gain on corporate and debt restructuring (...,11,13151,0.08,99.92
163,CHANGE__Gain on corporate and debt restructuri...,11,13151,0.08,99.92
164,CHANGE__Premium/discount on debt instruments,9,13153,0.07,99.93
165,CHANGE__Sales tax,8,13154,0.06,99.94
166,PL__Fair value gain on retained interest in a ...,6,13156,0.05,99.95
167,CHANGE__Fair value gain on retained interest i...,6,13156,0.05,99.95
168,PL__Of which: Income from carbon credits,5,13157,0.04,99.96
169,CHANGE__Of which: Income from carbon credits,5,13157,0.04,99.96
170,CHANGE__Consultancy fees to auditors,4,13158,0.03,99.97
171,PL__Profit on securitisation/assignment of ass...,2,13160,0.02,99.98



COMPANY COVERAGE DISTRIBUTION


,Available_Features
count,"13,162.00"
mean,10.33
std,26.62
min,0.00
25%,0.00
50%,0.00
75%,0.00
max,107.00



COMPANIES WITH THE MOST AVAILABLE FINANCIAL VARIABLES


,Company_Key,Company_Name,Available_Features,Missing_Features,Coverage_Percent
10259,SALZER ELECTRONICS LTD,Salzer Electronics Ltd.,107,75,58.79
5309,INDIAN OIL CORPN LTD,Indian Oil Corpn. Ltd.,107,75,58.79
12912,WELSPUN LIVING LTD,Welspun Living Ltd.,104,78,57.14
6363,KIRLOSKAR OIL ENGINES LTD,Kirloskar Oil Engines Ltd.,104,78,57.14
9277,R & B DENIMS LTD,R & B Denims Ltd.,104,78,57.14
13148,ZUARI INDUSTRIES LTD,Zuari Industries Ltd.,103,79,56.59
10672,SHIVA TEXYARN LTD,Shiva Texyarn Ltd.,103,79,56.59
11814,TATA POWER CO LTD,Tata Power Co. Ltd.,103,79,56.59
3361,ELECON ENGINEERING CO LTD,Elecon Engineering Co. Ltd.,103,79,56.59
1868,BIRLA CORPORATION LTD,Birla Corporation Ltd.,102,80,56.04



Dataset quality analysis saved.
